# ANALÍTICA EMPRESARIAL INTEGRADA
## LABORATORIO DIRIGIDO N.° 10 — Pronóstico de demanda

**Semana 10 · TECSUP 2026-II**

**Estudiante:** Trilce Aranda Cuevas

**Docente:** PILAR ROCIO SAYAN MEJIA
**Dataset real:** Online Retail II — UCI Machine Learning Repository (ID 502)
**Objetivo:** construir un pronóstico semanal de ventas, seleccionar modelo y política de cantidad con validación temporal sin información futura, evaluarlos una sola vez y decidir si conviene adoptarlos.

> El notebook termina en una **decisión fundamentada** (adoptar, mantener en evaluación o no adoptar), no solo en el modelo con menor error.

## Trabajo individual y entrega

Esta práctica dirigida es **individual**. Ejecute el notebook en **su propio Google Colab**, verifique los resultados y responda con sus propias interpretaciones. No se requiere informe separado: las respuestas, el cierre ejecutivo y las conclusiones quedan en este notebook.

**Al final de la sesión:**
1. Suba este notebook, ejecutado y con los retos resueltos, a la carpeta `semana10/` de su repositorio personal de GitHub.
2. Conserve la versión en su propio Google Colab y compártala con permiso de lectura.
3. En la **actividad de la semana** presente **ambos URLs**: el enlace directo al archivo en GitHub y el enlace a su Colab.


# 1. Pregunta de negocio

**¿Cuántas unidades debería planificar la empresa para las próximas semanas, qué riesgo asume si el pronóstico se equivoca y conviene adoptar el modelo?**

Para responder esta pregunta, se desarrollarán cuatro pasos que cumplen funciones diferentes:

1. **Pronóstico de ventas:** estima cuántas unidades se venderán cada semana.
2. **Política de cantidad:** establece cuántas unidades conviene tener disponibles, considerando el pronóstico y un posible colchón de seguridad.
3. **Evaluación económica:** calcula el costo de tener unidades sobrantes o no contar con suficientes unidades para cubrir la demanda.
4. **Decisión de adopción:** determina si conviene utilizar el modelo, mantenerlo en evaluación o no adoptarlo, comparándolo con un modelo de referencia (*baseline*).

## Diseño del laboratorio

| Elemento | Definición |
|---|---|
| Unidad temporal | Semana de lunes a domingo, identificada por la fecha del lunes. |
| Semanas válidas | Se consideran únicamente semanas completas según la cobertura del archivo. |
| Variable objetivo | Unidades vendidas por semana de un producto (SKU) del Reino Unido. |
| Test final | Últimas 8 semanas completas, evaluadas una sola vez. |
| Selección del SKU | Se realiza utilizando criterios calculados únicamente con información anterior al test. |
| Validación | *Backtesting rolling-origin* con un horizonte de 8 semanas dentro del periodo de entrenamiento. |
| Modelos | Naïve, Naïve estacional (m = 4) y *Exponential Smoothing*. |
| Regla de selección | Se consideran los modelos cuyo costo promedio sea menor o igual al costo mínimo más un margen del 1 %. Si el costo mínimo es cero, solo se consideran los modelos con costo cero. Los empates se resuelven por menor MAE, menor desviación del costo y menor complejidad. |
| Política de colchón | Se prueban multiplicadores de 0, 0.5, 1 y 1.5 veces el MAE previo. Este MAE se calcula utilizando únicamente errores anteriores a cada origen de pronóstico. |

## Resultado esperado

Al finalizar el laboratorio, se seleccionará el modelo y la política de cantidad mediante validación temporal. Después, ambos se evaluarán una sola vez con el test final y se compararán con el *baseline*.

La conclusión no dependerá únicamente de cuál modelo tenga menor error, sino también de sus costos económicos y de si demuestra una mejora suficiente frente al modelo de referencia.



## Actividad 1 — Revisión de conceptos

| Concepto | Definición con sus propias palabras y ejemplo | Pregunta de apoyo |
|---|---|---|
| Serie temporal | Es un conjunto de datos ordenados cronológicamente que permite analizar cómo cambia una variable con el tiempo. Por ejemplo, registrar las ventas de una tienda cada semana para pronosticar las ventas futuras. | **¿Por qué no se pueden barajar sus filas?** Porque se perdería el orden del tiempo y se podría utilizar información futura para predecir el pasado, generando una evaluación poco realista. |
| Semana completa | Es un periodo de siete días, de lunes a domingo, con información disponible para toda la semana. Esto permite comparar las ventas entre semanas de manera más justa. Por ejemplo, una semana con siete días registrados se puede comparar con otra semana completa. | **¿Por qué se excluye la primera semana del archivo?** Porque empieza un martes y no contiene los siete días, por lo que sus ventas no serían directamente comparables con las de una semana completa. |
| Tendencia y estacionalidad | La tendencia indica si los datos aumentan o disminuyen de forma sostenida, mientras que la estacionalidad representa un patrón que se repite en determinados periodos. Por ejemplo, las ventas pueden crecer con el tiempo y aumentar cada diciembre por las fiestas. | **¿Basta con que la serie suba y baje para afirmar estacionalidad?** No, porque los cambios deben seguir un patrón que se repita en periodos similares y no ser variaciones ocasionales. |
| Baseline (Naïve y Naïve estacional) | Es un modelo de referencia que permite comparar si otros métodos pronostican mejor. El Naïve utiliza el último valor observado, mientras que el Naïve estacional utiliza el valor de un periodo equivalente del ciclo anterior. | **¿Qué significa que un modelo complejo no supere al Naïve?** Significa que no demuestra una mejora frente a un método sencillo, por lo que no necesariamente vale la pena utilizar uno más complejo. |
| Exponential Smoothing | Es un método de pronóstico que da mayor importancia a los datos recientes y menos a los antiguos. Algunas variantes también consideran la tendencia y la estacionalidad para representar mejor el comportamiento de las ventas. | **¿Más componentes garantizan un mejor pronóstico?** No, porque agregar componentes aumenta la complejidad, pero solo conviene hacerlo si los resultados de la validación mejoran. |
| Partición temporal (train, validación, test) | Consiste en dividir los datos respetando su orden cronológico. El entrenamiento sirve para ajustar el modelo, la validación para comparar y seleccionar alternativas, y el test para evaluar el resultado final con datos no utilizados en la selección. | **¿Por qué el test se mira una sola vez?** Para conservar una evaluación independiente. Si se utiliza repetidamente para modificar el modelo, deja de ser una prueba imparcial. |
| Backtesting rolling-origin | Es una técnica que evalúa el pronóstico desde diferentes momentos del pasado. En cada origen se utiliza la información disponible hasta ese momento para predecir las semanas siguientes y luego se repite el procedimiento avanzando en el tiempo. | **¿Qué aporta frente a un único holdout?** Permite comprobar si el modelo funciona en distintos periodos y si sus resultados son consistentes, en lugar de depender de una sola evaluación. |
| MAE, RMSE y WAPE | Son métricas para medir los errores del pronóstico. El MAE calcula el error absoluto promedio en unidades, el RMSE da más importancia a los errores grandes y el WAPE expresa el error absoluto total como porcentaje de la demanda total. | **¿Cuál traduce el error directamente a unidades?** El MAE es el más fácil de interpretar porque indica cuántas unidades se equivoca el pronóstico en promedio. El RMSE también se expresa en unidades, pero penaliza más los errores grandes. |
| Costo asimétrico del error | Se presenta cuando equivocarse por exceso o por defecto genera costos diferentes. Por ejemplo, tener productos sobrantes puede ocasionar gastos de almacenamiento, mientras que quedarse sin productos puede significar ventas perdidas. | **¿Por qué dos errores de 20 unidades pueden costar distinto?** Porque el costo de las unidades sobrantes puede ser diferente al costo de las unidades faltantes. Por eso, no basta con medir el error: también hay que evaluar sus consecuencias económicas. |
| Pronóstico, política de cantidad y evaluación económica | El pronóstico estima las ventas futuras; la política de cantidad determina cuánto inventario conviene tener, incluyendo un posible colchón de seguridad; y la evaluación económica calcula los costos de los excedentes y faltantes para apoyar la decisión. | **¿La cantidad objetivo es lo que hay que comprar?** No. Primero se resta el inventario utilizable y los pedidos que llegarán a tiempo. La cantidad que falta después de ese cálculo es la que se necesita comprar. |


In [32]:
%pip install -q ucimlrepo statsmodels plotly scikit-learn openpyxl

In [33]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import sklearn
import statsmodels

from ucimlrepo import fetch_ucirepo
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import mean_absolute_error, mean_squared_error

# No se ocultan advertencias globalmente: las de cada ajuste se registran y se revisan al final.
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
VERSIONES = {"python": sys.version.split()[0], "pandas": pd.__version__, "numpy": np.__version__,
             "statsmodels": statsmodels.__version__, "scikit-learn": sklearn.__version__}
print("Versiones:", VERSIONES)

Versiones: {'python': '3.13.16', 'pandas': '2.2.3', 'numpy': '2.1.3', 'statsmodels': '0.15.0', 'scikit-learn': '1.6.1'}


# 2. Descarga de datos reales y trazabilidad

Se intenta primero la API oficial de UCI; si falla, el archivo ZIP oficial, del que se leen y concatenan sus **dos hojas**; si ambas fallan, se sube manualmente el archivo descargado desde https://archive.ics.uci.edu/dataset/502. La celda registra fuente, hojas, dimensiones, rango de fechas y versiones.

> Que otra copia tenga el mismo número de filas **no demuestra** que sea idéntica. Con otras versiones de las bibliotecas, los ajustes de Exponential Smoothing pueden producir cifras ligeramente distintas.

In [5]:
from io import BytesIO
from urllib.request import urlopen
from zipfile import ZipFile

def leer_dos_hojas(fuente_excel):
    hojas = pd.read_excel(fuente_excel, sheet_name=None)  # "Year 2009-2010" y "Year 2010-2011"
    return pd.concat(hojas.values(), ignore_index=True), list(hojas)

df, fuente, hojas_leidas = None, None, None
try:
    retail = fetch_ucirepo(id=502)
    df, fuente, hojas_leidas = retail.data.original.copy(), "API ucimlrepo (UCI ID 502)", "tabla única entregada por la API"
except Exception as exc:
    print("API ucimlrepo no disponible:", type(exc).__name__)

if df is None:
    try:
        url = "https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip"
        with urlopen(url, timeout=120) as respuesta:
            blob = respuesta.read()
        with ZipFile(BytesIO(blob)) as z:
            nombre = next(n for n in z.namelist() if n.lower().endswith(".xlsx"))
            df, hojas_leidas = leer_dos_hojas(BytesIO(z.read(nombre)))
        fuente = "ZIP oficial de UCI"
    except Exception as exc:
        print("ZIP oficial no disponible:", type(exc).__name__)

if df is None:
    locales = sorted(f for f in os.listdir(".")
                     if "online" in f.lower() and "retail" in f.lower() and f.lower().endswith((".xlsx", ".csv")))
    if not locales:
        try:
            from google.colab import files
            print("Suba el archivo online_retail_II.xlsx descargado desde UCI.")
            locales = list(files.upload().keys())
        except ImportError:
            raise FileNotFoundError("Coloque online_retail_II.xlsx en la carpeta de trabajo y vuelva a ejecutar.")
    archivo = locales[0]
    if archivo.lower().endswith(".xlsx"):
        df, hojas_leidas = leer_dos_hojas(archivo)
    else:
        df = pd.read_csv(archivo, dtype={"Invoice": str, "InvoiceNo": str, "StockCode": str})
        hojas_leidas = "no aplica (CSV)"
    fuente = f"archivo local: {archivo}"

fechas_raw = pd.to_datetime(df[[c for c in df.columns if "date" in str(c).lower()][0]], errors="coerce")
print("Fuente utilizada:", fuente)
print("Hojas:", hojas_leidas)
print("Dimensión:", df.shape)
print("Rango de fechas:", fechas_raw.min(), "→", fechas_raw.max())
print("Ejecución:", pd.Timestamp.now(tz="UTC").isoformat())
print("Versiones:", VERSIONES)
display(df.head())

API ucimlrepo no disponible: DatasetNotFoundError
Fuente utilizada: ZIP oficial de UCI
Hojas: ['Year 2009-2010', 'Year 2010-2011']
Dimensión: (1067371, 8)
Rango de fechas: 2009-12-01 07:45:00 → 2011-12-09 12:50:00
Ejecución: 2026-10-22T02:55:45.825923+00:00
Versiones: {'python': '3.13.16', 'pandas': '2.2.3', 'numpy': '2.1.3', 'statsmodels': '0.15.0', 'scikit-learn': '1.6.1'}


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [6]:
def find_col(columns, candidates):
    normalized = {str(c).lower().replace(" ", "").replace("_", ""): c for c in columns}
    for cand in candidates:
        key = cand.lower().replace(" ", "").replace("_", "")
        if key in normalized:
            return normalized[key]
    return None

mapping = {
    "InvoiceNo": find_col(df.columns, ["InvoiceNo", "Invoice"]),
    "StockCode": find_col(df.columns, ["StockCode"]),
    "Description": find_col(df.columns, ["Description"]),
    "Quantity": find_col(df.columns, ["Quantity"]),
    "InvoiceDate": find_col(df.columns, ["InvoiceDate"]),
    "UnitPrice": find_col(df.columns, ["UnitPrice", "Price"]),
    "CustomerID": find_col(df.columns, ["Customer ID", "CustomerID"]),
    "Country": find_col(df.columns, ["Country"]),
}
faltan = [k for k, v in mapping.items() if v is None and k not in ("Description", "CustomerID")]
if faltan:
    raise ValueError(f"No se identificaron columnas obligatorias: {faltan}")
df = df.rename(columns={v: k for k, v in mapping.items() if v is not None})
print(mapping)

{'InvoiceNo': 'Invoice', 'StockCode': 'StockCode', 'Description': 'Description', 'Quantity': 'Quantity', 'InvoiceDate': 'InvoiceDate', 'UnitPrice': 'Price', 'CustomerID': 'Customer ID', 'Country': 'Country'}


# 3. Control de calidad

Los `StockCode` nulos se descartan **antes** de convertir a texto; de lo contrario, la ausencia se convertiría en un producto llamado «nan».

In [7]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["UnitPrice"] = pd.to_numeric(df["UnitPrice"], errors="coerce")
print("StockCode nulos descartados:", int(df["StockCode"].isna().sum()))
df = df[df["StockCode"].notna()].copy()
df["StockCode"] = df["StockCode"].astype(str).str.strip()
df = df[df["StockCode"] != ""]
cancel = df["InvoiceNo"].astype(str).str.upper().str.startswith("C")

print("Fecha mínima:", df["InvoiceDate"].min(), "·", df["InvoiceDate"].min().day_name())
print("Fecha máxima:", df["InvoiceDate"].max(), "·", df["InvoiceDate"].max().day_name())
print("Cancelaciones:", int(cancel.sum()))
print("Quantity <= 0:", int((df["Quantity"] <= 0).sum()))
print("UnitPrice <= 0:", int((df["UnitPrice"] <= 0).sum()))
print("Fechas nulas:", int(df["InvoiceDate"].isna().sum()))

clean = df.loc[(~cancel) & df["InvoiceDate"].notna() & (df["Quantity"] > 0) &
               (df["UnitPrice"] > 0) & df["Country"].notna()].copy()
uk = clean[clean["Country"].astype(str).str.upper().eq("UNITED KINGDOM")].copy()
print("Filas válidas UK:", len(uk), "· SKUs UK:", uk["StockCode"].nunique())

StockCode nulos descartados: 0
Fecha mínima: 2009-12-01 07:45:00 · Tuesday
Fecha máxima: 2011-12-09 12:50:00 · Friday
Cancelaciones: 19494
Quantity <= 0: 22950
UnitPrice <= 0: 6207
Fechas nulas: 0
Filas válidas UK: 958501 · SKUs UK: 4905


### Pregunta 1
¿Por qué no sería correcto sumar directamente `Quantity` sin eliminar cancelaciones y cantidades no positivas?


**Respuesta:**

No sería correcto sumar directamente `Quantity` porque los registros de cancelaciones y las cantidades iguales o menores que cero pueden reducir o distorsionar el total de ventas. Primero se deben eliminar las cancelaciones y los registros con cantidades o precios no positivos para calcular la demanda registrada de forma más consistente. Así, el pronóstico se basa en ventas válidas y no en operaciones que no representan ventas efectivas.


# 4. Semanas completas y partición temporal

Las semanas van de **lunes a domingo** y se identifican por la fecha del lunes. Una semana es **completa** solo si el archivo cubre sus siete días: se excluyen la primera semana si el archivo no empieza en lunes y la última si no termina en domingo. Las **últimas 8 semanas completas** se reservan como test final.

Algunas semanas no tienen **ninguna venta registrada** en los datos filtrados. El cierre de la tienda es una **posible explicación no verificada**: los datos no lo confirman.

In [8]:
def lunes(fechas):
    fechas = pd.to_datetime(fechas)
    return fechas.dt.normalize() - pd.to_timedelta(fechas.dt.weekday, unit="D")

uk["week"] = lunes(uk["InvoiceDate"])
inicio_archivo, fin_archivo = df["InvoiceDate"].min(), df["InvoiceDate"].max()
lunes_inicio = inicio_archivo.normalize() - pd.Timedelta(days=inicio_archivo.weekday())
lunes_fin = fin_archivo.normalize() - pd.Timedelta(days=fin_archivo.weekday())
primera_completa = lunes_inicio if inicio_archivo.weekday() == 0 else lunes_inicio + pd.Timedelta(weeks=1)
ultima_completa = lunes_fin if fin_archivo.weekday() == 6 else lunes_fin - pd.Timedelta(weeks=1)

semanas = pd.date_range(primera_completa, ultima_completa, freq="W-MON")
TEST_H = 8
semanas_test = semanas[-TEST_H:]
semanas_train = semanas[:-TEST_H]

registros = uk.groupby("week").size().reindex(semanas, fill_value=0)
semanas_sin_registro = registros[registros == 0].index

print("Semanas excluidas por incompletas:", lunes_inicio.date(), "y", lunes_fin.date())
print("Semanas completas:", len(semanas), "·", semanas[0].date(), "→", semanas[-1].date())
print("Entrenamiento:", len(semanas_train), "semanas ·", semanas_train[0].date(), "→", semanas_train[-1].date())
print("Test final:", len(semanas_test), "semanas ·", semanas_test[0].date(), "→", semanas_test[-1].date())
print("Semanas sin ventas registradas en los datos filtrados (cierre: explicación posible no verificada):",
      [d.date() for d in semanas_sin_registro])

Semanas excluidas por incompletas: 2009-11-30 y 2011-12-05
Semanas completas: 104 · 2009-12-07 → 2011-11-28
Entrenamiento: 96 semanas · 2009-12-07 → 2011-10-03
Test final: 8 semanas · 2011-10-10 → 2011-11-28
Semanas sin ventas registradas en los datos filtrados (cierre: explicación posible no verificada): [datetime.date(2009, 12, 28), datetime.date(2010, 12, 27)]


### Pregunta 2
¿Por qué una partición aleatoria con `train_test_split` sería incorrecta en este problema?


**Respuesta:**

Una partición aleatoria sería incorrecta porque mezclaría semanas pasadas y futuras entre el entrenamiento y el test. Esto podría provocar que el modelo utilice información futura para predecir ventas anteriores y mostrar resultados poco realistas. Por eso, los datos deben dividirse cronológicamente, utilizando primero las semanas antiguas para entrenar y reservando las últimas 8 semanas completas para la evaluación final.


# 5. Selección reproducible del producto (solo con el entrenamiento)

Criterios, calculados **exclusivamente con las semanas de entrenamiento**:
1. **Periodo con ventas registradas cercano al entrenamiento completo:** primera venta dentro de las **4 primeras** semanas y última venta dentro de las **4 últimas**. El criterio tolera hasta 3 semanas iniciales o finales sin ventas, que luego se completan con cero aunque podrían corresponder a un periodo sin disponibilidad. **No verifica inventario.**
2. **Cobertura:** ventas registradas en al menos el 90 % de las semanas de entrenamiento con algún registro en la tienda.
3. Entre los elegibles, el de **mayor cantidad de unidades**; desempate por más semanas con ventas.

Se comprueba que haya al menos un producto elegible (y al menos dos para el Reto 3).

In [9]:
uk_train = uk[uk["week"].isin(semanas_train)]
semanas_con_registro_train = [w for w in semanas_train if w not in set(semanas_sin_registro)]

cobertura = (uk_train.groupby(["StockCode", "week"])["Quantity"].sum().reset_index()
             .groupby("StockCode")
             .agg(semanas_con_venta=("week", "nunique"), unidades=("Quantity", "sum"),
                  primera_venta=("week", "min"), ultima_venta=("week", "max"))
             .reset_index())
cobertura["cobertura"] = cobertura["semanas_con_venta"] / len(semanas_con_registro_train)
elegibles = cobertura[
    (cobertura["primera_venta"] <= semanas_train[3]) &      # dentro de las 4 primeras semanas
    (cobertura["ultima_venta"] >= semanas_train[-4]) &      # dentro de las 4 últimas semanas
    (cobertura["cobertura"] >= 0.90)
].sort_values(["unidades", "semanas_con_venta"], ascending=False).reset_index(drop=True)

if len(elegibles) == 0:
    raise ValueError("Ningún SKU cumple los criterios: revise la carga y la limpieza antes de continuar.")
print("SKUs elegibles:", len(elegibles), "· suficientes para el Reto 3:", len(elegibles) >= 2)
SKU = elegibles.loc[0, "StockCode"]
desc = uk.loc[uk["StockCode"] == SKU, "Description"].dropna().astype(str).mode().iloc[0]
print("SKU seleccionado:", SKU, "·", desc)
display(elegibles.head(5))

SKUs elegibles: 389 · suficientes para el Reto 3: True
SKU seleccionado: 84077 · WORLD WAR 2 GLIDERS ASSTD DESIGNS


,StockCode,semanas_con_venta,unidades,primera_venta,ultima_venta,cobertura
0,84077,93,88273,2009-12-07,2011-10-03,0.989362
1,85123A,94,80903,2009-12-07,2011-10-03,1.000000
2,85099B,94,79134,2009-12-07,2011-10-03,1.000000
3,21212,94,67878,2009-12-07,2011-10-03,1.000000
4,17003,88,67514,2009-12-07,2011-10-03,0.936170


# 6. Serie semanal de ventas registradas

**Qué representa un cero.** Un cero indica que **no hay ventas registradas** del SKU en esa semana dentro de los datos filtrados. Los datos no permiten distinguir entre:
- **disponibilidad de inventario** con demanda nula;
- **demanda no satisfecha** por falta de inventario (no queda registrada);
- **ausencia de registros** (por ejemplo, semanas sin ninguna venta registrada en la tienda).

Completar con cero es un **supuesto sobre las ventas observadas**: no demuestra demanda cero ni disponibilidad continua. Por eso el laboratorio pronostica **ventas registradas**, que solo aproximan la demanda.

In [10]:
serie = (uk[uk["StockCode"] == SKU].groupby("week")["Quantity"].sum()
         .reindex(semanas, fill_value=0).astype(float))
serie.name = "unidades"
train = serie.loc[semanas_train]
test = serie.loc[semanas_test]

ceros = serie[serie == 0].index
print("Semanas de la serie:", len(serie), "· semanas sin ventas registradas del SKU:", len(ceros))
print("  coinciden con semanas sin ningún registro en la tienda:", [d.date() for d in ceros if d in set(semanas_sin_registro)])
print("  semanas con registros en la tienda pero sin ventas del SKU:", [d.date() for d in ceros if d not in set(semanas_sin_registro)])
print("Unidades en entrenamiento:", int(train.sum()))

Semanas de la serie: 104 · semanas sin ventas registradas del SKU: 3
  coinciden con semanas sin ningún registro en la tienda: [datetime.date(2009, 12, 28), datetime.date(2010, 12, 27)]
  semanas con registros en la tienda pero sin ventas del SKU: [datetime.date(2011, 1, 17)]
Unidades en entrenamiento: 88273


# 7. Exploración (solo con el entrenamiento)

La exploración que orienta decisiones usa únicamente el entrenamiento: mirar el test antes de elegir el modelo sería una fuga de información. En este laboratorio las medias móviles de las **últimas 4 y 8 semanas** se usan para **explorar** el nivel; no se evalúan como método de pronóstico (podrían usarse como tal, pero no forman parte de esta comparación).

In [11]:
explora = pd.DataFrame({"ventas": train})
explora["media_movil_4"] = explora["ventas"].rolling(4).mean()
explora["media_movil_8"] = explora["ventas"].rolling(8).mean()
fig = go.Figure()
for col, nombre in [("ventas", "Ventas registradas"), ("media_movil_4", "Media móvil 4"), ("media_movil_8", "Media móvil 8")]:
    fig.add_trace(go.Scatter(x=explora.index, y=explora[col], name=nombre))
fig.update_layout(title=f"Entrenamiento · SKU {SKU} · {desc}", xaxis_title="Semana (lunes)", yaxis_title="Unidades")
fig.show()

perfil = (explora.assign(semana_anio=explora.index.isocalendar().week.astype(int))
          .groupby("semana_anio")["ventas"].mean().reset_index())
fig = px.line(perfil, x="semana_anio", y="ventas", markers=True,
              title="Perfil promedio por semana del año (entrenamiento)",
              labels={"semana_anio": "Semana del año", "ventas": "Ventas promedio"})
fig.show()

anual = train.groupby(train.index.year).agg(["count", "mean", "median", "max"])
display(anual.round(1))

,count,mean,median,max
2009,4,493.0,384.0,1204.0
2010,52,1034.4,608.0,7063.0
2011,40,812.8,558.0,3216.0


### Pregunta 3
Identifique evidencia de tendencia, estacionalidad o quiebres en el entrenamiento. Diferencie lo que observa de lo que todavía necesita comprobar, y explique qué representa un cero en la serie.


**Respuesta:**

En el entrenamiento, las ventas registradas del producto presentan variaciones a lo largo del tiempo. Las medias móviles de 4 y 8 semanas ayudan a observar si existe una tendencia de aumento o disminución, mientras que el perfil por semana del año permite explorar posibles patrones estacionales. Sin embargo, estas gráficas son indicios y no suficientes para confirmar la estacionalidad; sería necesario comprobar si los patrones se repiten de forma consistente. También se deben revisar posibles quiebres o cambios bruscos en las ventas.

Un cero significa que no se registraron ventas del producto en esa semana. Esto no demuestra que la demanda haya sido nula, ya que pudo no haber inventario disponible o no haber registros de ventas. Por eso, la serie representa las ventas registradas y no necesariamente toda la demanda real.


# 8. Modelos, cantidad operativa, costo y regla de selección

- **Naïve:** repite la última semana observada.
- **Naïve estacional (m = 4):** repite la semana de hace 4 semanas. El periodo de 4 semanas es un **supuesto de modelado** (ciclo operativo corto), no una propiedad comprobada del negocio; con 96 semanas de entrenamiento no hay suficientes años para estimar con confianza un ciclo anual de 52 semanas.
- **Exponential Smoothing:** nivel y tendencia aditivos con estacionalidad aditiva m = 4. Si un entrenamiento tuviera menos de 2 × 4 + 8 = 16 semanas, se simplifica a nivel y tendencia (documentado; se informa si ocurre).

**Cantidad operativa:** todo pronóstico se convierte en `max(redondeo(pronóstico), 0)` y **todas** las métricas y costos se calculan sobre esa cantidad.

**Costo base (supuesto académico, no proviene de UCI):** £0,40 por unidad excedente y £1,50 por unidad faltante.

**Regla única de selección** (se usa para el modelo, la política de colchón, el Reto 1 y el Reto 3):
1. Candidatos: costo promedio de validación ≤ mínimo × 1,01 (**incluye** exactamente el 1 %; se admite una holgura de 1e-9 por redondeo numérico). Si el mínimo es 0, solo son candidatos los costos iguales a 0.
2. Desempate entre candidatos: menor MAE promedio, luego menor desviación del costo y, al final, menor complejidad.

In [12]:
SEASON = 4
OVER_COST, SHORT_COST = 0.40, 1.50
MODELOS = ["Naive", "SeasonalNaive", "ExpSmoothing"]   # en orden de complejidad
TOLERANCIA = 0.01
registro_ajustes = []

def ajustar_es(y, etiqueta):
    """Ajusta Exponential Smoothing y registra todas las advertencias del ajuste."""
    if len(y) >= 2 * SEASON + 8:
        modelo = ExponentialSmoothing(y, trend="add", seasonal="add", seasonal_periods=SEASON,
                                      initialization_method="estimated")
        especificacion = "nivel + tendencia + estacionalidad aditivas (m = 4)"
    else:
        modelo = ExponentialSmoothing(y, trend="add", seasonal=None, initialization_method="estimated")
        especificacion = "simplificado: nivel + tendencia"
    with warnings.catch_warnings(record=True) as avisos:
        warnings.simplefilter("always")
        ajuste = modelo.fit(optimized=True)
    registro_ajustes.append({"etapa": etiqueta.split(" · ")[0], "ajuste": etiqueta, "n_train": len(y),
                             "especificacion": especificacion, "advertencias": len(avisos),
                             "tipos": "; ".join(sorted({w.category.__name__ for w in avisos})),
                             "mensajes": " | ".join(sorted({str(w.message)[:110] for w in avisos}))})
    return ajuste

def pronosticar(modelo, y, h, etiqueta=""):
    if modelo == "Naive":
        f = np.repeat(y.iloc[-1], h)
    elif modelo == "SeasonalNaive":
        f = np.resize(y.iloc[-SEASON:].values, h)
    elif modelo == "ExpSmoothing":
        f = np.asarray(ajustar_es(y, etiqueta).forecast(h), dtype=float)
    else:
        raise ValueError(modelo)
    return np.maximum(np.round(f), 0)   # cantidad operativa

def evaluar(real, cantidad, over=OVER_COST, short=SHORT_COST):
    real, cantidad = np.asarray(real, float), np.asarray(cantidad, float)
    exceso, faltante = np.maximum(cantidad - real, 0), np.maximum(real - cantidad, 0)
    return {"MAE": mean_absolute_error(real, cantidad),
            "RMSE": mean_squared_error(real, cantidad) ** 0.5,
            "WAPE": np.abs(real - cantidad).sum() / real.sum() if real.sum() > 0 else np.nan,  # no definido si la demanda total es 0
            "unidades_excedentes": exceso.sum(), "unidades_faltantes": faltante.sum(),
            "costo": (exceso * over + faltante * short).sum()}

def resumir(filas, por, orden):
    """Resume los folds por modelo o por política. 'orden' va de la opción más simple a la más compleja."""
    t = (pd.DataFrame(filas).groupby(por)
           .agg(costo_prom=("costo", "mean"), costo_sd=("costo", "std"),
                MAE_prom=("MAE", "mean"), MAE_sd=("MAE", "std"), RMSE_prom=("RMSE", "mean"),
                WAPE_prom=("WAPE", "mean"), excedentes=("unidades_excedentes", "sum"),
                faltantes=("unidades_faltantes", "sum"))
           .reindex(orden))
    t["complejidad"] = range(len(orden))
    return t

def seleccionar(tabla, tol=TOLERANCIA):
    """Regla única de selección (ver el texto de esta sección)."""
    minimo = tabla["costo_prom"].min()
    limite = minimo * (1 + tol)          # si minimo == 0, limite == 0: solo empatan costos 0
    candidatos = tabla[tabla["costo_prom"] <= limite + 1e-9]
    return candidatos.sort_values(["MAE_prom", "costo_sd", "complejidad"], kind="stable").index[0]
print("Funciones listas.")

Funciones listas.


# 9. Backtesting rolling-origin (solo con el entrenamiento)

Cada fold entrena con todo lo anterior a su origen y pronostica las **8 semanas** siguientes, igual que el test final. Se exige al menos 52 semanas de entrenamiento; los bloques de validación no se superponen y el último termina justo antes del test. La celda **comprueba** que ningún fold invada el test.

In [13]:
BT_H, MIN_TRAIN = 8, 52
origenes = list(range(len(train) - BT_H, MIN_TRAIN - 1, -BT_H))[::-1]

def backtest(y, modelos, etiqueta):
    """Pronostica cada bloque de validación desde su origen. Devuelve métricas por fold y cantidades."""
    filas, cantidades = [], {}
    for k, origen in enumerate(origenes, start=1):
        tr, va = y.iloc[:origen], y.iloc[origen:origen + BT_H]
        assert tr.index[-1] < va.index[0] and va.index[-1] < semanas_test[0], "un fold invade el test"
        for m in modelos:
            cantidades[(k, m)] = pronosticar(m, tr, BT_H, f"{etiqueta} · fold {k}")
            filas.append({**evaluar(va, cantidades[(k, m)]), "fold": k, "modelo": m})
    return filas, cantidades

filas_bt, q_bt = backtest(train, MODELOS, "backtesting")
bt = pd.DataFrame(filas_bt)
folds = pd.DataFrame([{"fold": k, "entrenamiento": f"{train.index[0].date()} → {train.index[o - 1].date()}", "n_train": o,
                       "validacion": f"{train.index[o].date()} → {train.index[o + BT_H - 1].date()}"}
                      for k, o in enumerate(origenes, start=1)])
display(folds)
display(bt.round(2))
print("Comprobado: los", len(origenes), "folds terminan antes del test (", semanas_test[0].date(), ").")

,fold,entrenamiento,n_train,validacion
0,1,2009-12-07 → 2010-12-27,56,2011-01-03 → 2011-02-21
1,2,2009-12-07 → 2011-02-21,64,2011-02-28 → 2011-04-18
2,3,2009-12-07 → 2011-04-18,72,2011-04-25 → 2011-06-13
3,4,2009-12-07 → 2011-06-13,80,2011-06-20 → 2011-08-08
4,5,2009-12-07 → 2011-08-08,88,2011-08-15 → 2011-10-03


,MAE,RMSE,WAPE,unidades_excedentes,unidades_faltantes,costo,fold,modelo
0,608.12,1044.62,1.00,0.0,4865.0,7297.5,1,Naive
1,528.88,758.45,0.87,1195.0,3036.0,5032.0,1,SeasonalNaive
2,1599.12,1838.12,2.63,11475.0,1318.0,6567.0,1,ExpSmoothing
3,1111.12,1460.84,0.83,113.0,8776.0,13209.2,2,Naive
4,789.88,1000.51,0.59,1317.0,5002.0,8029.8,2,SeasonalNaive
5,949.12,1147.68,0.71,4251.0,3342.0,6713.4,2,ExpSmoothing
6,723.88,824.92,0.77,4550.0,1241.0,3681.5,3,Naive
7,1248.88,1463.50,1.33,8557.0,1434.0,5573.8,3,SeasonalNaive
8,850.75,960.03,0.91,5384.0,1422.0,4286.6,3,ExpSmoothing
9,391.88,471.10,0.75,47.0,3088.0,4650.8,4,Naive


Comprobado: los 5 folds terminan antes del test ( 2011-10-10 ).


In [14]:
resumen_val = resumir(filas_bt, "modelo", MODELOS)
SELECCIONADO = seleccionar(resumen_val)
display(resumen_val.round(2))
print("MODELO SELECCIONADO EN VALIDACIÓN (fijado antes de mirar el test):", SELECCIONADO)
fig = px.box(bt, x="modelo", y="costo", points="all", title="Costo simulado por fold de validación (£)",
             labels={"modelo": "Modelo", "costo": "Costo por fold (£)"})
fig.show()

,costo_prom,costo_sd,MAE_prom,MAE_sd,RMSE_prom,WAPE_prom,excedentes,faltantes,complejidad
modelo,,,,,,,,,
Naive,6239.22,4296.41,620.2,327.68,819.13,0.75,5469.0,19339.0,0
SeasonalNaive,4950.34,2065.83,666.5,370.02,830.61,0.83,13853.0,12807.0,1
ExpSmoothing,4324.10,2335.11,927.3,420.72,1063.63,1.29,30925.0,6167.0,2


MODELO SELECCIONADO EN VALIDACIÓN (fijado antes de mirar el test): ExpSmoothing


### Pregunta 4
¿Qué modelo se seleccionó en validación y con qué regla? ¿Coinciden el ranking por costo y el ranking por MAE o RMSE? ¿Por qué el ranking por WAPE no aporta un criterio adicional al MAE?


**Respuesta:**

El modelo seleccionado fue **Exponential Smoothing**, porque obtuvo el menor costo promedio en la validación, con £4,324.10. La regla de selección considera los modelos cuyo costo promedio esté dentro del 1 % del costo mínimo y, entre ellos, prioriza el menor MAE, la menor desviación del costo y la menor complejidad. En este caso, Exponential Smoothing fue seleccionado por tener el menor costo promedio.

Los rankings por costo y por MAE o RMSE no necesariamente coinciden, porque estas métricas miden aspectos diferentes: el costo considera las consecuencias económicas de los excedentes y faltantes, mientras que el MAE mide el error promedio y el RMSE penaliza más los errores grandes. Por eso, un modelo puede tener mayor error, pero generar un menor costo económico.

El WAPE no aporta un criterio adicional al MAE para ordenar los modelos en los mismos periodos, porque ambos se basan en el error absoluto y el WAPE es el error absoluto total dividido entre la demanda total. Si la demanda total es la misma para todos los modelos, ambos producen el mismo ranking.


### Pregunta 5
¿Qué tan estable fue cada modelo entre folds? Use la desviación del costo y del MAE.


**Respuesta:**

La estabilidad se evalúa observando la desviación del costo y del MAE entre los folds. **Naïve** presentó la mayor desviación del costo (£4,296.41) y del MAE (327.68), por lo que sus resultados fueron más variables. **SeasonalNaive** fue más estable en costo, con una desviación de £2,065.83, y tuvo una desviación del MAE de 370.02. **Exponential Smoothing** obtuvo una desviación del costo de £2,335.11 y la mayor desviación del MAE, de 420.72. En conclusión, SeasonalNaive fue el más estable en costos, mientras que Naïve mostró menor variación del MAE. Esto demuestra que un modelo puede ser más estable en una métrica y menos estable en otra.


# 10. Política de colchón calibrada sin información futura (antes del test)

El colchón suma unidades al pronóstico para protegerse de faltantes: **colchón = ⌈multiplicador × MAE previo⌉**. Se comparan los multiplicadores 0; 0,5; 1 y 1,5.

**Regla de calibración (sin errores futuros).** Para cada origen, el *MAE previo* se calcula con el modelo seleccionado en los **2 bloques de 8 semanas inmediatamente anteriores** a ese origen: cada bloque se pronostica entrenando solo con lo anterior a él. Así, el colchón del fold 1 usa solo errores anteriores al fold 1.
- Si no hay historia suficiente (algún bloque quedaría con menos de 16 semanas de entrenamiento), el colchón de ese origen es **0** y se informa. Con 52 semanas mínimas no ocurre en este laboratorio.
- La política (modelo, multiplicador y regla) se elige con la **regla única de selección** y queda **fijada antes del test**.
- Para el test, el MAE previo se calcula con la misma regla desde el final del entrenamiento (solo información previa al test).
- Después de evaluar, la **magnitud** del colchón se actualiza con la historia completa para proyectar, **sin cambiar la regla**.

In [15]:
MULTIPLICADORES = [0, 0.5, 1.0, 1.5]
BLOQUES_CAL = 2

def mae_previo(modelo, y, etiqueta, bloques=BLOQUES_CAL):
    """MAE del modelo en los 'bloques' bloques de 8 semanas que terminan al final de y.
    Solo usa y (información hasta el origen). Devuelve None si la historia no alcanza."""
    errores = []
    for b in range(bloques, 0, -1):
        corte = len(y) - b * BT_H
        if corte < 2 * SEASON + 8:
            return None
        real = y.iloc[corte:corte + BT_H].values
        errores.append(np.abs(real - pronosticar(modelo, y.iloc[:corte], BT_H, f"{etiqueta} · bloque {b}")))
    return float(np.mean(np.concatenate(errores)))

def colchon(mult, mae):
    return 0 if (mae is None or mult == 0) else int(np.ceil(mult * mae))

def comparar_politicas(bloques=BLOQUES_CAL, etiqueta="calibración"):
    filas = []
    for k, origen in enumerate(origenes, start=1):
        tr, va = train.iloc[:origen], train.iloc[origen:origen + BT_H]
        mae_k = mae_previo(SELECCIONADO, tr, f"{etiqueta} · fold {k}", bloques)
        for mult in MULTIPLICADORES:
            c = colchon(mult, mae_k)
            filas.append({**evaluar(va, q_bt[(k, SELECCIONADO)] + c), "fold": k, "multiplicador": mult,
                          "mae_previo": mae_k, "colchon": c,
                          "ultima_semana_calibracion": tr.index[-1].date(), "inicio_validacion": va.index[0].date()})
    return pd.DataFrame(filas)

pol_folds = comparar_politicas()
assert (pol_folds["ultima_semana_calibracion"] < pol_folds["inicio_validacion"]).all(), "colchón con información futura"
resumen_pol = resumir(pol_folds, "multiplicador", MULTIPLICADORES)
MULT = seleccionar(resumen_pol)

display(pol_folds.loc[pol_folds["multiplicador"] == 1.0,
                      ["fold", "ultima_semana_calibracion", "inicio_validacion", "mae_previo"]].round(1))
display(resumen_pol.round(2))
MAE_TEST = mae_previo(SELECCIONADO, train, "calibración test")       # solo entrenamiento
COLCHON_TEST = colchon(MULT, MAE_TEST)
POLITICA = f"{SELECCIONADO} + {MULT:g} × MAE previo"
print("POLÍTICA FIJADA ANTES DEL TEST:", POLITICA)
print(f"Colchón para el test: {COLCHON_TEST} unidades por semana (MAE previo = {MAE_TEST:.1f}, calculado solo con el entrenamiento)")

,fold,ultima_semana_calibracion,inicio_validacion,mae_previo
2,1,2010-12-27,2011-01-03,1400.7
6,2,2011-02-21,2011-02-28,1533.2
10,3,2011-04-18,2011-04-25,1274.1
14,4,2011-06-13,2011-06-20,899.9
18,5,2011-08-08,2011-08-15,821.3


,costo_prom,costo_sd,MAE_prom,MAE_sd,RMSE_prom,WAPE_prom,excedentes,faltantes,complejidad
multiplicador,,,,,,,,,
0.0,4324.10,2335.11,927.30,420.72,1063.63,1.29,30925.0,6167.0,0
0.5,4806.72,1784.08,1334.35,475.19,1477.53,1.88,50934.0,2440.0,1
1.0,5833.74,1661.61,1812.80,521.90,1988.40,2.53,72363.0,149.0,2
1.5,7674.08,2000.88,2398.15,625.28,2537.15,3.31,95926.0,0.0,3


POLÍTICA FIJADA ANTES DEL TEST: ExpSmoothing + 0 × MAE previo
Colchón para el test: 0 unidades por semana (MAE previo = 618.8, calculado solo con el entrenamiento)


### Pregunta 6
¿Qué política de colchón quedó fijada antes del test y por qué? Explique por qué el colchón de cada fold se calcula solo con errores anteriores a su origen.


**Respuesta:**

La política fijada antes del test fue **Exponential Smoothing sin colchón adicional (multiplicador 0)**, porque obtuvo el menor costo promedio en la validación, con £4,324.10 por fold. Aunque agregar un colchón reduce las unidades faltantes, también aumenta los excedentes y el costo total, por lo que en este caso no resultó conveniente.

El colchón de cada fold se calcula únicamente con errores anteriores a su origen para evitar utilizar información futura. Así, la política se evalúa en condiciones similares a las que tendría la empresa al pronosticar ventas reales, manteniendo una validación temporal justa y confiable.


# 11. Evaluación final en test (una sola vez)

Se reentrena cada modelo con las 96 semanas de entrenamiento y se pronostican las 8 semanas de test. Se evalúan los tres modelos sin colchón y la **política fijada**. Si otra opción resulta mejor aquí, se informa, pero **no** cambia la selección.

In [16]:
cantidades_test = {m: pronosticar(m, train, TEST_H, "test") for m in MODELOS}
q_politica = cantidades_test[SELECCIONADO] + COLCHON_TEST
filas = [{**evaluar(test, q), "opcion": f"{m} (sin colchón)"} for m, q in cantidades_test.items()]
if MULT > 0:
    filas.append({**evaluar(test, q_politica), "opcion": f"Política fijada: {POLITICA}"})
res_test = pd.DataFrame(filas).set_index("opcion")
display(res_test.round(2))

fila_pol = f"Política fijada: {POLITICA}" if MULT > 0 else f"{SELECCIONADO} (sin colchón)"
costo_pol, costo_naive = res_test.loc[fila_pol, "costo"], res_test.loc["Naive (sin colchón)", "costo"]
diferencia = costo_naive - costo_pol        # > 0 ahorro; < 0 sobrecosto
print("Política evaluada:", POLITICA)
if SELECCIONADO == "Naive" and MULT == 0:
    print("La política seleccionada es el propio Naïve: no hay diferencia que calcular frente a él.")
elif costo_naive == 0:
    print(f"Diferencia frente a Naïve: £{diferencia:.2f}. No se calcula porcentaje porque el costo de Naïve es cero.")
elif diferencia > 0:
    print(f"Ahorro frente a Naïve: £{diferencia:.2f} ({diferencia / costo_naive * 100:.1f} %).")
else:
    print(f"Sobrecosto frente a Naïve: £{-diferencia:.2f} ({-diferencia / costo_naive * 100:.1f} %).")
mejor_retro = res_test["costo"].idxmin()
if mejor_retro != fila_pol:
    print(f"Retrospectivamente, «{mejor_retro}» habría costado menos en test; la selección no cambia.")

fig = go.Figure()
fig.add_trace(go.Scatter(x=test.index, y=test.values, mode="lines+markers", name="Ventas reales", line=dict(width=3)))
for m, q in cantidades_test.items():
    fig.add_trace(go.Scatter(x=test.index, y=q, mode="lines+markers", name=m))
fig.update_layout(title="Test final: ventas reales y cantidades operativas", xaxis_title="Semana (lunes)", yaxis_title="Unidades")
fig.show()

,MAE,RMSE,WAPE,unidades_excedentes,unidades_faltantes,costo
opcion,,,,,,
Naive (sin colchón),678.75,1545.12,0.48,961.0,4469.0,7087.9
SeasonalNaive (sin colchón),773.75,1631.81,0.54,641.0,5549.0,8579.9
ExpSmoothing (sin colchón),776.62,1700.86,0.54,329.0,5884.0,8957.6


Política evaluada: ExpSmoothing + 0 × MAE previo
Sobrecosto frente a Naïve: £1869.70 (26.4 %).
Retrospectivamente, «Naive (sin colchón)» habría costado menos en test; la selección no cambia.


### Pregunta 7
¿La política seleccionada cumplió en test? Compare su costo con los baselines e indique el ahorro o sobrecosto frente a Naïve. ¿El modelo con menor RMSE fue también el de menor costo?


**Respuesta:**

La política seleccionada, **Exponential Smoothing sin colchón**, no cumplió en el test, ya que obtuvo un costo de £5,884, mientras que el modelo Naïve alcanzó un costo de £4,469. Esto representa un **sobrecosto de £1,415, equivalente al 31.7 %** respecto al Naïve. Por lo tanto, el modelo seleccionado en validación no demostró una mejora económica en la evaluación final.

El modelo con menor RMSE tampoco fue el de menor costo. Naïve obtuvo el menor RMSE (1,545.12) y el menor costo (£4,469), mientras que Exponential Smoothing presentó un RMSE de 1,700.86 y un costo de £5,884. Esto demuestra que es importante evaluar tanto los errores del pronóstico como sus consecuencias económicas antes de decidir si conviene adoptar un modelo.


### Diagnóstico del pico del test (retrospectivo)

Se identifica la semana de mayor venta del test y el pedido más grande de esa semana. Un pedido extraordinario **aislado** es un **pico puntual**; un **quiebre estructural** exige un cambio persistente de nivel o patrón, que 8 semanas no permiten confirmar. Los modelos usados son **univariantes**: solo ven las ventas pasadas del SKU y no disponían de información específica sobre ese pedido.

La evaluación principal **conserva** esa semana. El cálculo sin ella es un **análisis retrospectivo de sensibilidad**, no una evaluación alternativa.

In [17]:
semana_pico = test.idxmax()
lineas_pico = uk[(uk["StockCode"] == SKU) & (uk["week"] == semana_pico)]
pedidos = lineas_pico.groupby(["InvoiceNo", "CustomerID"], dropna=False)["Quantity"].sum().sort_values(ascending=False)
print(f"Semana de mayor venta en test: {semana_pico.date()} · {test.max():.0f} unidades")
factura, cliente = pedidos.index[0]
cliente = "sin dato" if pd.isna(cliente) else str(cliente).removesuffix(".0")
print(f"Pedido más grande: factura {factura} · cliente {cliente} · {pedidos.iloc[0]:.0f} unidades "
      f"({pedidos.iloc[0] / test.max() * 100:.1f} % de la semana)")
print("Mediana semanal del entrenamiento:", train.median(), "unidades")

sin_pico = test.index != semana_pico
sens = pd.DataFrame({m: evaluar(test[sin_pico], cantidades_test[m][sin_pico]) for m in MODELOS}).T
if MULT > 0:
    sens.loc["Política fijada"] = evaluar(test[sin_pico], q_politica[sin_pico])
print("\nANÁLISIS RETROSPECTIVO DE SENSIBILIDAD (7 semanas, sin la semana del pico). No reemplaza la evaluación principal.")
display(sens[["MAE", "unidades_excedentes", "unidades_faltantes", "costo"]].round(2))

Semana de mayor venta en test: 2011-10-24 · 5328 unidades
Pedido más grande: factura 573008 · cliente 12901 · 4800 unidades (90.1 % de la semana)
Mediana semanal del entrenamiento: 584.5 unidades

ANÁLISIS RETROSPECTIVO DE SENSIBILIDAD (7 semanas, sin la semana del pico). No reemplaza la evaluación principal.


,MAE,unidades_excedentes,unidades_faltantes,costo
Naive,155.86,961.0,130.0,579.4
SeasonalNaive,232.86,641.0,989.0,1739.9
ExpSmoothing,205.86,329.0,1112.0,1799.6


### Pregunta 8
¿Qué error fue más grave en el test: sobreestimar o subestimar? Use las unidades y los costos. ¿El pico del test indica un quiebre estructural o un pico puntual? ¿Qué muestra, y qué no, el análisis de sensibilidad sin esa semana?


**Respuesta:**

En el test, el error más costoso fue **subestimar las ventas**, porque cada unidad faltante cuesta £1.50, mientras que cada unidad excedente cuesta £0.40. Por ejemplo, Naïve tuvo 961 unidades excedentes y 130 faltantes, con un costo total de £579.40 al excluir retrospectivamente la semana del pico. Esto muestra que el costo depende tanto de la cantidad de unidades equivocadas como del tipo de error.

El pico ocurrió la semana del 24 de octubre de 2011, con 5,328 unidades vendidas. El pedido más grande representó el 90.1 % de las ventas de esa semana, lo que sugiere un **pico puntual asociado a un pedido extraordinario**, no evidencia suficiente para afirmar un quiebre estructural. El análisis de sensibilidad sin esa semana muestra cómo cambian los errores y costos cuando se excluye el pico, pero no reemplaza la evaluación principal ni demuestra que el modelo vaya a funcionar mejor en el futuro.


# 12. Reentrenamiento final y proyección académica

Después de cerrar la evaluación se reentrena el modelo seleccionado con **toda** la historia (104 semanas) y se actualiza la **magnitud** del colchón con la misma regla (los 2 bloques más recientes, que ahora incluyen el test). Se muestra también la proyección de Naïve como referencia.

La tabla es una **proyección académica**, no una orden de compra:
- La **cantidad objetivo semanal** es un objetivo para cada semana. El laboratorio **no simula el arrastre de inventario** entre semanas: si una semana sobra mercadería, esta reduce la necesidad de la siguiente. Por eso la suma de los objetivos semanales no equivale a la cantidad a comprar.
- Una compra simplificada para una semana sería `max(objetivo − inventario utilizable − pedidos que llegarán a tiempo, 0)`: nunca negativa, y solo cuenta el inventario apto para la venta y los pedidos que llegarán antes de esa semana. El laboratorio no dispone de esos datos.

In [18]:
FINAL_H = 8
pron_final = pronosticar(SELECCIONADO, serie, FINAL_H, "final")
pron_naive = pronosticar("Naive", serie, FINAL_H, "final")
MAE_FINAL = mae_previo(SELECCIONADO, serie, "calibración final")   # misma regla, historia completa
COLCHON_FINAL = colchon(MULT, MAE_FINAL)
semanas_futuras = pd.date_range(serie.index[-1] + pd.Timedelta(weeks=1), periods=FINAL_H, freq="W-MON")
plan = pd.DataFrame({"Semana": semanas_futuras.date,
                     f"Pronóstico {SELECCIONADO}": pron_final.astype(int),
                     "Colchón (política fijada)": COLCHON_FINAL,
                     "Cantidad objetivo semanal": (pron_final + COLCHON_FINAL).astype(int),
                     "Referencia Naïve": pron_naive.astype(int)})
display(plan)
print(f"Política aplicada: {POLITICA} · MAE previo actualizado = {MAE_FINAL:.1f} · colchón = {COLCHON_FINAL} unidades por semana")
print("Proyección académica: no es una orden de compra ni descuenta inventario ni pedidos en tránsito.")

,Semana,Pronóstico ExpSmoothing,Colchón (política fijada),Cantidad objetivo semanal,Referencia Naïve
0,2011-12-05,1075,0,1075,689
1,2011-12-12,899,0,899,689
2,2011-12-19,1617,0,1617,689
3,2011-12-26,998,0,998,689
4,2012-01-02,1103,0,1103,689
5,2012-01-09,927,0,927,689
6,2012-01-16,1645,0,1645,689
7,2012-01-23,1027,0,1027,689


Política aplicada: ExpSmoothing + 0 × MAE previo · MAE previo actualizado = 611.1 · colchón = 0 unidades por semana
Proyección académica: no es una orden de compra ni descuenta inventario ni pedidos en tránsito.


# 13. Decisión de adopción y resumen automático

La decisión se toma con una regla fijada de antemano:
- **Adoptar** (como piloto controlado): la política seleccionada costó menos que Naïve en el test **y** en la mayoría de los folds de validación.
- **Mantener en evaluación:** superó a Naïve en el test, pero no de forma consistente en validación.
- **No adoptar como sustituto de Naïve:** en el test costó igual o más que Naïve. Naïve sigue como referencia; el modelo puede seguir evaluándose sin decidir compras con él.

Un **piloto** sirve para comprobar, durante varias semanas nuevas y con la misma regla, si la política reduce el costo frente a Naïve; no justifica por sí mismo una cantidad de compra.

In [19]:
naive_bt = bt[bt["modelo"] == "Naive"].set_index("fold")["costo"]
pol_bt = pol_folds[pol_folds["multiplicador"] == MULT].set_index("fold")["costo"]
folds_ganados = int((pol_bt < naive_bt).sum())
if costo_pol < costo_naive and folds_ganados > len(origenes) / 2:
    DECISION = "Adoptar como piloto controlado frente a Naïve"
elif costo_pol < costo_naive:
    DECISION = "Mantener en evaluación"
else:
    DECISION = "No adoptar como sustituto de Naïve; Naïve se mantiene como referencia"

resumen = {
    "Producto analizado": f"{SKU} · {desc}",
    "Seleccionado antes del test": f"{SELECCIONADO}; política {POLITICA} (regla única de selección)",
    "Costo promedio de validación de la política": f"£{resumen_pol.loc[MULT, 'costo_prom']:.2f} ± £{resumen_pol.loc[MULT, 'costo_sd']:.2f} ({len(origenes)} folds)",
    "Folds de validación en que superó a Naïve": f"{folds_ganados} de {len(origenes)}",
    "Resultado en test (8 semanas)": f"£{costo_pol:.2f} frente a £{costo_naive:.2f} de Naïve",
    "Frente a Naïve": ("ahorro" if diferencia > 0 else "sobrecosto") + f" de £{abs(diferencia):.2f}" + (f" ({abs(diferencia) / costo_naive * 100:.1f} %)" if costo_naive > 0 else ""),
    "Unidades excedentes / faltantes en test": f"{res_test.loc[fila_pol, 'unidades_excedentes']:.0f} / {res_test.loc[fila_pol, 'unidades_faltantes']:.0f}",
    "Decisión": DECISION,
    "Proyección académica (suma de objetivos semanales)": f"{int(plan['Cantidad objetivo semanal'].sum())} unidades; no es una orden de compra",
}
for k, v in resumen.items():
    print(f"{k}: {v}")

Producto analizado: 84077 · WORLD WAR 2 GLIDERS ASSTD DESIGNS
Seleccionado antes del test: ExpSmoothing; política ExpSmoothing + 0 × MAE previo (regla única de selección)
Costo promedio de validación de la política: £4324.10 ± £2335.11 (5 folds)
Folds de validación en que superó a Naïve: 4 de 5
Resultado en test (8 semanas): £8957.60 frente a £7087.90 de Naïve
Frente a Naïve: sobrecosto de £1869.70 (26.4 %)
Unidades excedentes / faltantes en test: 329 / 5884
Decisión: No adoptar como sustituto de Naïve; Naïve se mantiene como referencia
Proyección académica (suma de objetivos semanales): 9291 unidades; no es una orden de compra


### Pregunta 9 — Cierre ejecutivo
Cierre ejecutivo (máximo 12 líneas). Distinga: modelo y política seleccionados antes del test; resultado de su evaluación frente a Naïve; decisión (adoptar, mantener en evaluación o no adoptar) y su fundamento; proyección académica y por qué no es una orden de compra; una condición que cambiaría la decisión; y las limitaciones (supuestos de costos, ceros, disponibilidad y un solo test de 8 semanas).


**Respuesta:**

Se seleccionó **Exponential Smoothing sin colchón adicional**, porque obtuvo el menor costo promedio en validación: £4,324.10. Sin embargo, en el test de 8 semanas su costo fue de £8,957.60, superior al de Naïve (£7,087.90), con un sobrecosto de £1,869.70 (26.4 %).

Por ello, la decisión es **no adoptarlo como sustituto de Naïve**, que se mantiene como referencia. Aunque el modelo seleccionado superó a Naïve en 4 de los 5 folds de validación, no logró una mejora económica en el test final.

La proyección académica es de 9,291 unidades objetivo para las próximas 8 semanas, pero no constituye una orden de compra porque no descuenta el inventario disponible ni los pedidos pendientes que llegarán a tiempo.

La decisión podría cambiar si nuevas evaluaciones muestran que el modelo reduce los costos frente a Naïve de manera consistente. Entre las limitaciones están los costos asumidos, la interpretación de las semanas con cero ventas, la falta de información sobre disponibilidad de inventario y la evaluación con un único test de 8 semanas.


# 14. Retos individuales de aplicación

La parte guiada ya desarrolló los procedimientos. En esta sección los aplicará a tres casos empresariales. Cada reto tiene cuatro apartados: **A** plantear o preparar el análisis, **B** implementar y obtener resultados, **C** comparar e interpretar la evidencia y **D** formular una decisión justificada con sus limitaciones. Reutilice las funciones de la parte guiada (`evaluar`, `resumir`, `seleccionar`, `backtest`, `pronosticar`, `mae_previo`, `colchon` y `comparar_politicas`) y los objetos ya calculados (`train`, `test`, `origenes`, `q_bt`, `cantidades_test`, `elegibles`). No modifique datos, semanas, pronósticos ni el test. Las respuestas deben ser breves, pero suficientes para justificar la decisión con cifras.

> **Entrega.** «Resuelva los tres retos y sus apartados A, B, C y D en su notebook personal. Guarde la versión ejecutada, con resultados e interpretaciones, en la carpeta semana10/ de su repositorio personal de GitHub y en su propio Google Colab. En la actividad de la semana, en el espacio habilitado, presente ambos enlaces: GitHub y Colab. Verifique que permitan la revisión del trabajo.»

## Reto 1 — Una campaña cambia el costo de los faltantes

**Caso.** La empresa prepara una campaña comercial. Durante la campaña, quedarse sin producto significa perder ventas y clientes que llegaron por la promoción, mientras que guardar unidades sobrantes cuesta menos. El responsable de abastecimiento necesita saber si la recomendación del modelo cambia cuando el faltante se vuelve más costoso.

**Problema.** ¿Se elegiría el mismo método de pronóstico bajo distintos costos del error y, con la evidencia del test, cuál debería usar abastecimiento como referencia?

**Datos, supuestos y recursos**
- Producto y semanas del laboratorio; pronósticos de validación ya calculados (`q_bt`) y cantidades de test (`cantidades_test`).
- Escenarios de costo (supuestos académicos, no provienen de UCI): A faltante £1,00 y exceso £0,50; B faltante £1,50 y exceso £0,40 (base); C faltante £3,00 y exceso £0,25.
- Regla común de selección (`resumir` y `seleccionar`). No se modifican ventas ni pronósticos para favorecer un resultado.

**Apartados**
- **A.** Plantear los escenarios y la decisión
- **B.** Calcular los costos de validación y seleccionar
- **C.** Evaluar en test y comparar frente a Naïve
- **D.** Recomendar a abastecimiento

**Evidencias que deben quedar en el notebook**
- Tabla de escenarios con la razón faltante / exceso.
- Tabla de costos promedio de validación por escenario y modelo, con el seleccionado.
- Tabla de test con excedentes, faltantes, costo y diferencia frente a Naïve, y su gráfico.
- Interpretaciones A–D.

### Reto 1 · Apartado A — Plantear los escenarios y la decisión
Construya una tabla con los tres escenarios y la razón faltante / exceso. Explique qué representa cada escenario para el negocio y formule la decisión empresarial que se tomará con el análisis.

In [21]:
# Escenarios de costos: (faltante, exceso), en £ por unidad
ESCENARIOS = {
    "A": (1.00, 0.50),
    "B": (1.50, 0.40),
    "C": (3.00, 0.25)
}

r1_escenarios = pd.DataFrame([
    {
        "Escenario": esc,
        "Costo por faltante (£)": short,
        "Costo por exceso (£)": over,
        "Faltante / exceso": round(short / over, 2)
    }
    for esc, (short, over) in ESCENARIOS.items()
])

display(r1_escenarios)

,Escenario,Costo por faltante (£),Costo por exceso (£),Faltante / exceso
0,A,1.0,0.50,2.00
1,B,1.5,0.40,3.75
2,C,3.0,0.25,12.00


**Interpretación 1A:**


Los escenarios representan distintas condiciones económicas. En A, el faltante cuesta el doble que el exceso; en B, que es el escenario base, el faltante cuesta 3.75 veces más; y en C, el faltante cuesta 12 veces más, por lo que quedarse sin producto resulta mucho más costoso.

La decisión consiste en identificar qué modelo tiene el menor costo de validación en cada escenario y comprobar si mantiene una ventaja frente a Naïve en el test. Así se determinará qué método conviene utilizar como referencia para abastecimiento.


### Reto 1 · Apartado B — Calcular los costos de validación y seleccionar
Para cada escenario, recalcule el costo de cada fold con los pronósticos ya disponibles (`q_bt`), resuma por modelo y aplique la regla común de selección. Presente el costo promedio de validación de los tres modelos y el seleccionado en cada escenario.

In [34]:
r1_val = {}
r1_eleccion = {}

for esc, (short, over) in ESCENARIOS.items():
    filas = []

    for k, origen in enumerate(origenes, start=1):
        real = train.iloc[origen:origen + BT_H]

        for m in MODELOS:
            cantidad = q_bt[(k, m)]

            resultado = evaluar(
                real,
                cantidad,
                over=over,
                short=short
            )

            filas.append({
                **resultado,
                "fold": k,
                "modelo": m
            })

    r1_val[esc] = resumir(filas, "modelo", MODELOS)
    r1_eleccion[esc] = seleccionar(r1_val[esc])

# Tabla de costos promedio de validación
tabla_r1b = pd.DataFrame({
    esc: r1_val[esc]["costo_prom"]
    for esc in ESCENARIOS
}).rename_axis("Modelo")

tabla_r1b["Seleccionado"] = [
    "Sí" if modelo in r1_eleccion.values() else "No"
    for modelo in tabla_r1b.index
]

display(tabla_r1b.round(2))

print("Modelo seleccionado por escenario:")
for esc in ESCENARIOS:
    print(
        f"Escenario {esc}: {r1_eleccion[esc]} "
        f"(costo promedio £{r1_val[esc].loc[r1_eleccion[esc], 'costo_prom']:.2f})"
    )

,A,B,C,Seleccionado
Modelo,,,,
Naive,4414.7,6239.22,11876.85,No
SeasonalNaive,3946.7,4950.34,8376.85,Sí
ExpSmoothing,4325.9,4324.10,5246.45,Sí


Modelo seleccionado por escenario:
Escenario A: SeasonalNaive (costo promedio £3946.70)
Escenario B: ExpSmoothing (costo promedio £4324.10)
Escenario C: ExpSmoothing (costo promedio £5246.45)


**Interpretación 1B:**


La tabla permite comparar el costo promedio de cada modelo bajo diferentes costos por faltantes y excedentes. Cuando el faltante se vuelve más costoso, puede cambiar el modelo seleccionado, porque la evaluación económica penaliza más las unidades que no alcanzan a cubrir las ventas. La selección se realiza con la misma regla en los tres escenarios, considerando también el MAE y la variabilidad del costo cuando corresponde.


### Reto 1 · Apartado C — Evaluar en test y comparar frente a Naïve
Evalúe en test el modelo seleccionado en cada escenario (la selección ya quedó definida con validación) y Naïve con los mismos costos. Construya una tabla con excedentes, faltantes, costo y diferencia frente a Naïve, y un gráfico de barras. Interprete qué escenario y qué tipo de error explican las diferencias.

In [35]:
# Evaluar en test el modelo seleccionado y Naïve
filas_r1c = []

for esc, (short, over) in ESCENARIOS.items():
    modelo = r1_eleccion[esc]

    # Evaluación del modelo seleccionado
    res_sel = evaluar(
        test,
        cantidades_test[modelo],
        over=over,
        short=short
    )

    # Evaluación de Naïve con los mismos costos
    res_naive = evaluar(
        test,
        cantidades_test["Naive"],
        over=over,
        short=short
    )

    filas_r1c.append({
        "Escenario": esc,
        "Modelo seleccionado": modelo,
        "Excedentes": res_sel["unidades_excedentes"],
        "Faltantes": res_sel["unidades_faltantes"],
        "Costo del seleccionado (£)": res_sel["costo"],
        "Costo de Naïve (£)": res_naive["costo"],
        # Positivo = ahorro frente a Naïve; negativo = sobrecosto
        "Diferencia vs Naïve (£)": res_naive["costo"] - res_sel["costo"]
    })

tabla_r1c = pd.DataFrame(filas_r1c).set_index("Escenario")

display(tabla_r1c.round(2))

# Gráfico comparativo de costos en test
grafico_r1c = tabla_r1c[
    ["Costo del seleccionado (£)", "Costo de Naïve (£)"]
].reset_index().melt(
    id_vars="Escenario",
    var_name="Modelo",
    value_name="Costo (£)"
)

fig = px.bar(
    grafico_r1c,
    x="Escenario",
    y="Costo (£)",
    color="Modelo",
    barmode="group",
    title="Reto 1: costo en test por escenario",
    labels={"Escenario": "Escenario de costos"}
)
fig.show()


,Modelo seleccionado,Excedentes,Faltantes,Costo del seleccionado (£),Costo de Naïve (£),Diferencia vs Naïve (£)
Escenario,,,,,,
A,SeasonalNaive,641.0,5549.0,5869.50,4949.50,-920.0
B,ExpSmoothing,329.0,5884.0,8957.60,7087.90,-1869.7
C,ExpSmoothing,329.0,5884.0,17734.25,13647.25,-4087.0


**Interpretación 1C:**


Los resultados permiten comparar el costo del modelo seleccionado con el de Naïve en cada escenario. Una diferencia positiva indica ahorro frente a Naïve, mientras que una diferencia negativa representa un sobrecosto.

Al aumentar el costo de los faltantes en relación con los excedentes, quedarse corto en la cantidad planificada tiene un mayor impacto económico. Para identificar qué escenario explica mejor las diferencias, se deben revisar conjuntamente los costos, las unidades excedentes y las unidades faltantes de la tabla. El gráfico permite observar en qué escenario cada modelo resulta más conveniente.


### Reto 1 · Apartado D — Recomendar a abastecimiento
Redacte una recomendación para abastecimiento: ¿cambia la selección entre escenarios?, ¿qué método usaría como referencia y por qué?, ¿qué límites tiene la conclusión? Distinga el ahorro simulado de un beneficio real demostrado.


**Decisión 1D:**


La selección cambia según los costos: en el escenario A se eligió **SeasonalNaive**, mientras que en B y C se seleccionó **Exponential Smoothing**. Esto ocurre porque, al aumentar el costo de los faltantes, cambia la importancia económica de los errores.

Sin embargo, en el test, los modelos seleccionados tuvieron un costo mayor que Naïve en los tres escenarios: £920 más en A, £1,869.70 más en B y £4,087 más en C. Por ello, recomendaría mantener **Naïve como referencia para abastecimiento**, ya que presentó un menor costo en la evaluación final.

Esta conclusión tiene limitaciones: los costos son supuestos académicos, el test comprende solo ocho semanas y las ventas registradas no necesariamente representan toda la demanda real. Además, estos resultados son simulaciones y no demuestran un beneficio económico real. Antes de aplicar una política de compra, convendría evaluarla con nuevos datos y considerar la disponibilidad de inventario.


## Reto 2 — ¿Conviene añadir un colchón?

**Caso.** El área comercial propone planificar unidades adicionales cada semana para reducir faltantes. Almacén advierte que el excedente también cuesta: espacio, capital inmovilizado y posibles liquidaciones. Se debe decidir con evidencia si conviene añadir un colchón al pronóstico.

**Problema.** ¿Qué política de cantidad, sin colchón o con colchón de 0,5; 1 o 1,5 veces el MAE previo, debería usarse, y qué costo produjo en test frente a no usar colchón?

**Datos, supuestos y recursos**
- Modelo seleccionado en la parte guiada (`SELECCIONADO`) y sus pronósticos de validación y test.
- Colchón = ⌈multiplicador × MAE previo⌉; el MAE previo usa solo los 2 bloques de 8 semanas anteriores a cada origen (`mae_previo`, `colchon`, `comparar_politicas`).
- Costos base: £1,50 por faltante y £0,40 por exceso (supuestos académicos). Regla común de selección.

**Apartados**
- **A.** Definir pronóstico, cantidad planificada y políticas
- **B.** Comparar las políticas en validación
- **C.** Seleccionar el multiplicador y evaluarlo en test
- **D.** Recomendar sobre el colchón

**Evidencias que deben quedar en el notebook**
- Tabla del MAE previo y del colchón de cada política por fold.
- Costos por fold y resumen por política con excedentes, faltantes y folds en que el colchón mejoró, más el gráfico de excedentes y faltantes.
- Tabla de test de la política seleccionada frente a la opción sin colchón.
- Interpretaciones A–D.

### Reto 2 · Apartado A — Definir pronóstico, cantidad planificada y políticas
Explique la diferencia entre pronóstico y cantidad planificada. Calcule, para cada fold, el MAE previo (solo con semanas anteriores a su origen) y el colchón que tendría cada política (0; 0,5; 1 y 1,5 × MAE previo).

In [36]:
# Preparar el MAE previo y el colchón por fold y política
filas_r2a = []

for k, origen in enumerate(origenes, start=1):
    historial = train.iloc[:origen]

    mae_k = mae_previo(
        SELECCIONADO,
        historial,
        f"Reto 2 · fold {k}"
    )

    for mult in MULTIPLICADORES:
        filas_r2a.append({
            "Fold": k,
            "Última semana de calibración": historial.index[-1].date(),
            "Inicio de validación": train.index[origen].date(),
            "MAE previo": mae_k,
            "Multiplicador": mult,
            "Colchón (unidades)": colchon(mult, mae_k)
        })

r2_prep = pd.DataFrame(filas_r2a)

# Comprobar que la calibración sea anterior a la validación
assert (
    pd.to_datetime(r2_prep["Última semana de calibración"])
    < pd.to_datetime(r2_prep["Inicio de validación"])
).all(), "Se detectó información futura en la calibración."

display(r2_prep.round(2))


,Fold,Última semana de calibración,Inicio de validación,MAE previo,Multiplicador,Colchón (unidades)
0,1,2010-12-27,2011-01-03,1400.69,0.0,0
1,1,2010-12-27,2011-01-03,1400.69,0.5,701
2,1,2010-12-27,2011-01-03,1400.69,1.0,1401
3,1,2010-12-27,2011-01-03,1400.69,1.5,2102
4,2,2011-02-21,2011-02-28,1533.25,0.0,0
5,2,2011-02-21,2011-02-28,1533.25,0.5,767
6,2,2011-02-21,2011-02-28,1533.25,1.0,1534
7,2,2011-02-21,2011-02-28,1533.25,1.5,2300
8,3,2011-04-18,2011-04-25,1274.12,0.0,0
9,3,2011-04-18,2011-04-25,1274.12,0.5,638



**Interpretación 2A:**

El MAE previo disminuye de 1,400.69 a 821.31 unidades entre los folds, lo que indica que el error reciente del modelo fue menor en los últimos periodos de calibración. El colchón se calcula multiplicando ese error por el factor elegido y redondeando hacia arriba. Por ejemplo, con un multiplicador de 0.5, el primer fold añade 701 unidades, mientras que con 1.5 añade 2,102 unidades. Todos los cálculos utilizan semanas anteriores a la validación, evitando información futura. Un colchón mayor puede reducir los faltantes, pero también aumentar los excedentes y sus costos.


### Reto 2 · Apartado B — Comparar las políticas en validación
Evalúe las cuatro políticas en los folds de validación con calibración temporal (`comparar_politicas`). Presente el costo por fold, el resumen por política (costo promedio, desviación, excedentes y faltantes), cuántos folds mejoró cada colchón frente a no usarlo y un gráfico de excedentes y faltantes.

In [37]:
# Evaluar las políticas con calibración temporal
r2_folds = comparar_politicas(etiqueta="reto2 · validación")

# Resumen por multiplicador
r2_resumen = resumir(
    r2_folds,
    "multiplicador",
    MULTIPLICADORES
)

# Costo por fold y multiplicador
tabla_r2_costos = r2_folds.pivot(
    index="fold",
    columns="multiplicador",
    values="costo"
).rename(columns={
    0.0: "Sin colchón",
    0.5: "0.5 × MAE",
    1.0: "1 × MAE",
    1.5: "1.5 × MAE"
})

display(tabla_r2_costos.round(2))

# Resumen: costos, variabilidad y unidades excedentes/faltantes
display(
    r2_resumen[
        [
            "costo_prom", "costo_sd", "MAE_prom",
            "excedentes", "faltantes"
        ]
    ].round(2)
)

# Contar folds en que cada colchón cuesta menos que no usarlo
costo_base = r2_folds[
    r2_folds["multiplicador"] == 0.0
].set_index("fold")["costo"]

mejoras = []
for mult in MULTIPLICADORES:
    costos_mult = r2_folds[
        r2_folds["multiplicador"] == mult
    ].set_index("fold")["costo"]

    mejoras.append({
        "Multiplicador": mult,
        "Folds con menor costo que sin colchón": int(
            (costos_mult < costo_base).sum()
        ),
        "Total de folds": len(costo_base)
    })

tabla_r2_mejoras = pd.DataFrame(mejoras)
display(tabla_r2_mejoras)

# Gráfico de unidades excedentes y faltantes por política
graf_r2 = r2_resumen[
    ["excedentes", "faltantes"]
].reset_index().melt(
    id_vars="multiplicador",
    var_name="Tipo de unidades",
    value_name="Unidades acumuladas"
)

fig = px.bar(
    graf_r2,
    x="multiplicador",
    y="Unidades acumuladas",
    color="Tipo de unidades",
    barmode="group",
    title="Reto 2: excedentes y faltantes por política",
    labels={
        "multiplicador": "Multiplicador del MAE previo",
        "Unidades acumuladas": "Unidades en validación"
    }
)
fig.show()


multiplicador,Sin colchón,0.5 × MAE,1 × MAE,1.5 × MAE
fold,,,,
1,6567.0,7478.3,8546.0,10789.2
2,6713.4,5345.0,5555.5,7723.6
3,4286.6,4563.1,5664.8,7703.2
4,2534.0,3974.0,5414.0,6854.0
5,1519.5,2673.2,3988.4,5300.4


,costo_prom,costo_sd,MAE_prom,excedentes,faltantes
multiplicador,,,,,
0.0,4324.10,2335.11,927.30,30925.0,6167.0
0.5,4806.72,1784.08,1334.35,50934.0,2440.0
1.0,5833.74,1661.61,1812.80,72363.0,149.0
1.5,7674.08,2000.88,2398.15,95926.0,0.0


,Multiplicador,Folds con menor costo que sin colchón,Total de folds
0,0.0,0,5
1,0.5,1,5
2,1.0,1,5
3,1.5,0,5


**Interpretación 2B:**

Los resultados muestran que **no añadir un colchón tuvo el menor costo promedio**, con £4,324.10. Al usar un colchón de 0.5 × MAE, el costo aumentó a £4,806.72; con 1 × MAE subió a £5,833.74, y con 1.5 × MAE llegó a £7,674.08.

Como se observa, al aumentar el colchón, los faltantes disminuyen de 6,167 unidades a cero, pero los excedentes aumentan de 30,925 a 95,926 unidades. Esto significa que se reduce el riesgo de quedarse sin stock, pero se generan más unidades sobrantes y mayores costos.

En cuanto a la comparación por folds, los colchones de 0.5 × MAE y 1 × MAE fueron más económicos que no usar colchón en solo uno de los cinco folds. El colchón de 1.5 × MAE no mejoró el costo en ningún fold. Aunque esta última política tuvo la menor desviación estándar del costo entre las cuatro políticas, no fue la más económica.

Con los costos establecidos, conviene mantener la política sin colchón, porque presenta el menor costo promedio. Sin embargo, si evitar faltantes fuera más importante que reducir el costo total, podría evaluarse un colchón mayor como alternativa.


### Reto 2 · Apartado C — Seleccionar el multiplicador y evaluarlo en test
Seleccione el multiplicador con la regla común (sin mirar el test). Calcule el colchón del test con el MAE previo del entrenamiento y evalúe la política seleccionada frente a la opción sin colchón: costo, excedentes y faltantes. Compruebe si coincide con la política fijada en la sección 10.

In [40]:
# Reto 2C: seleccionar el multiplicador sin mirar el test
r2_mult = seleccionar(r2_resumen)

# Calcular el MAE previo usando solo el entrenamiento
mae_t = mae_previo(SELECCIONADO, train, "reto2 · test")
colchon_t = colchon(r2_mult, mae_t)

print(f"Multiplicador seleccionado: {r2_mult} × MAE")
print(f"MAE previo del entrenamiento: {mae_t:.2f}")
print(f"Colchón para el test: {colchon_t} unidades")

# Comparar sin colchón frente a la política seleccionada
res_sin = evaluar(test, cantidades_test[SELECCIONADO])
res_sel = evaluar(
    test,
    cantidades_test[SELECCIONADO] + colchon_t
)

tabla_r2c = pd.DataFrame([res_sin, res_sel], index=["Sin colchón", f"Política seleccionada ({r2_mult} × MAE)"])
display(tabla_r2c.round(2))

# Comprobar coincidencia con la política de la sección 10
print(
    "¿Coincide con la sección 10?:",
    r2_mult == 0.0
)

Multiplicador seleccionado: 0.0 × MAE
MAE previo del entrenamiento: 618.75
Colchón para el test: 0 unidades


,MAE,RMSE,WAPE,unidades_excedentes,unidades_faltantes,costo
Sin colchón,776.62,1700.86,0.54,329.0,5884.0,8957.6
Política seleccionada (0.0 × MAE),776.62,1700.86,0.54,329.0,5884.0,8957.6


¿Coincide con la sección 10?: True


**Interpretación 2C:**


La regla de selección eligió un multiplicador de **0 × MAE**, por lo que el colchón para el test es de 0 unidades. El MAE previo del entrenamiento fue de 618.75 unidades.

Al comparar ambas políticas, los resultados son idénticos: un costo de £8,957.60, 329 unidades excedentes y 5,884 unidades faltantes. Esto ocurre porque la política seleccionada no añade unidades adicionales al pronóstico.

La política seleccionada coincide con la de la sección 10. Según los costos establecidos y los resultados de validación, no conviene añadir un colchón. Sin embargo, en el test se observa que los faltantes siguen siendo elevados, por lo que sería importante evaluar si se justifica asumir un mayor costo para reducirlos.


### Reto 2 · Apartado D — Recomendar sobre el colchón
Recomiende si corresponde añadir colchón y fundamente con la validación. Explique por qué la cantidad objetivo semanal no equivale automáticamente a una cantidad a comprar (inventario utilizable, pedidos que llegarán a tiempo y arrastre entre semanas).

**Decisión 2D:**

Se recomienda **no añadir un colchón por el momento**, porque en la validación la política sin colchón obtuvo el menor costo promedio (£4,324.10). Los colchones de 0.5 × MAE, 1 × MAE y 1.5 × MAE elevaron el costo promedio a £4,806.72, £5,833.74 y £7,674.08, respectivamente. Aunque los colchones redujeron los faltantes, también aumentaron los excedentes y el costo total.

Esta decisión debe revisarse si la empresa considera que evitar faltantes es más importante que minimizar el costo. En el test se registraron 5,884 unidades faltantes, por lo que existe un riesgo que debe evaluarse, aunque añadir colchón no resultó más económico según los costos establecidos.

Además, la **cantidad objetivo semanal no equivale automáticamente a la cantidad que se debe comprar**. Antes de realizar un pedido, se debe considerar el inventario utilizable disponible, descontar las unidades comprometidas y tener en cuenta los pedidos que llegarán a tiempo. También se debe considerar el arrastre entre semanas: las unidades sobrantes de una semana pueden cubrir parte de la necesidad de la siguiente.

Por tanto, el pronóstico y el colchón sirven como referencia para planificar, pero la compra final debe calcularse según la posición real del inventario, los plazos de entrega y las necesidades pendientes.


### Reto 2 · Ampliación opcional: ventana de calibración
Repita la comparación con 1, 2 y 3 bloques de calibración y observe si cambia el multiplicador elegido. Si muestra costos de test de las políticas descartadas, identifíquelos como retrospectivos: no se usan para cambiar la política.

In [41]:
# Ampliación opcional: comparar ventanas de calibración
ventanas = []

for b in [1, 2, 3]:
    folds_b = comparar_politicas(
        bloques=b,
        etiqueta=f"Reto 2 · calibración {b} bloque(s)"
    )

    resumen_b = resumir(
        folds_b, "multiplicador", MULTIPLICADORES
    )

    mult_b = seleccionar(resumen_b)

    ventanas.append({
        "Bloques de calibración": b,
        "Multiplicador elegido": mult_b,
        "Costo promedio": resumen_b.loc[mult_b, "costo_prom"],
        "Desviación del costo": resumen_b.loc[mult_b, "costo_sd"],
        "Excedentes": resumen_b.loc[mult_b, "excedentes"],
        "Faltantes": resumen_b.loc[mult_b, "faltantes"]
    })

    print(f"\nVentana: {b} bloque(s)")
    display(resumen_b.round(2))

ventanas = pd.DataFrame(ventanas)
display(ventanas.round(2))



Ventana: 1 bloque(s)


,costo_prom,costo_sd,MAE_prom,MAE_sd,RMSE_prom,WAPE_prom,excedentes,faltantes,complejidad
multiplicador,,,,,,,,,
0.0,4324.10,2335.11,927.30,420.72,1063.63,1.29,30925.0,6167.0,0
0.5,4744.00,1831.07,1310.35,497.63,1453.95,1.85,49910.0,2504.0,1
1.0,5704.00,1828.09,1764.35,578.40,1938.06,2.48,70310.0,264.0,2
1.5,7414.24,2311.82,2316.95,722.44,2459.48,3.21,92678.0,0.0,3



Ventana: 2 bloque(s)


,costo_prom,costo_sd,MAE_prom,MAE_sd,RMSE_prom,WAPE_prom,excedentes,faltantes,complejidad
multiplicador,,,,,,,,,
0.0,4324.10,2335.11,927.30,420.72,1063.63,1.29,30925.0,6167.0,0
0.5,4806.72,1784.08,1334.35,475.19,1477.53,1.88,50934.0,2440.0,1
1.0,5833.74,1661.61,1812.80,521.90,1988.40,2.53,72363.0,149.0,2
1.5,7674.08,2000.88,2398.15,625.28,2537.15,3.31,95926.0,0.0,3



Ventana: 3 bloque(s)


,costo_prom,costo_sd,MAE_prom,MAE_sd,RMSE_prom,WAPE_prom,excedentes,faltantes,complejidad
multiplicador,,,,,,,,,
0.0,4324.10,2335.11,927.30,420.72,1063.63,1.29,30925.0,6167.0,0
0.5,4870.40,1658.07,1342.15,423.83,1482.27,1.90,51070.0,2616.0,1
1.0,5927.90,1405.32,1823.80,420.91,1993.02,2.56,72535.0,417.0,2
1.5,7663.84,1417.40,2394.95,442.94,2540.88,3.32,95798.0,0.0,3


,Bloques de calibración,Multiplicador elegido,Costo promedio,Desviación del costo,Excedentes,Faltantes
0,1,0.0,4324.1,2335.11,30925.0,6167.0
1,2,0.0,4324.1,2335.11,30925.0,6167.0
2,3,0.0,4324.1,2335.11,30925.0,6167.0


**Comentario opcional 2:**


Al comparar ventanas de calibración de 1, 2 y 3 bloques, se observa si el multiplicador seleccionado se mantiene o cambia al utilizar más información histórica. La decisión se basa en el costo promedio de validación, aplicando la regla común de selección.

Si el multiplicador se mantiene, significa que la decisión es consistente con distintas ventanas de calibración. Si cambia, la política es sensible al periodo histórico utilizado y conviene considerar esa variabilidad antes de implementarla.

Los costos del test de las políticas descartadas, si se calculan, son únicamente retrospectivos. No deben utilizarse para modificar la selección, porque eso implicaría elegir la política usando información del periodo de evaluación.


## Reto 3 — ¿Funciona la misma solución para otro producto?

**Caso.** La empresa quiere extender el procedimiento a un segundo producto de alta rotación antes de aplicarlo a toda la cartera. La gerencia pregunta si el método que se eligió para el primer producto puede generalizarse.

**Problema.** ¿Qué modelo se selecciona para el segundo producto con las mismas reglas, cómo le fue en test frente a Naïve y qué haría la empresa con cada producto?

**Datos, supuestos y recursos**
- Segundo SKU de la lista de elegibles, definida solo con el entrenamiento (no se elige mirando el test).
- Mismas semanas completas, limpieza, backtesting, costos base y regla común de selección.
- Se comparan modelos **sin colchón** para mantener el alcance claro; la política de colchón se trabajó en el Reto 2.

**Apartados**
- **A.** Identificar el producto y describir su serie
- **B.** Backtesting y selección con la regla común
- **C.** Evaluar en test y comparar ambos productos
- **D.** Recomendación consolidada

**Evidencias que deben quedar en el notebook**
- Serie del segundo producto y tabla descriptiva del entrenamiento frente al primero, con su gráfico.
- Tabla de backtesting por modelo y seleccionado.
- Tabla comparativa de ambos productos con modelo, costos, ahorro o sobrecosto y resultado.
- Interpretaciones A–D.

### Reto 3 · Apartado A — Identificar el producto y describir su serie
Identifique el segundo SKU elegible, construya su serie con las mismas semanas completas y separe entrenamiento y test. Describa su comportamiento solo con el entrenamiento y compárelo con el primer producto (tabla y gráfico).

In [44]:

# Identificar el segundo SKU elegible sin mirar el test
assert len(elegibles) >= 2, "Se necesita un segundo SKU elegible"

SKU2 = elegibles.loc[1, "StockCode"]
print("Segundo SKU:", SKU2)

# Obtener la descripción del producto
desc2 = elegibles.loc[elegibles["StockCode"] == SKU2].copy()
display(desc2)

# Construir la serie semanal usando la misma función
# y el mismo procedimiento de la sección 6.
serie2 = (
    uk[uk["StockCode"] == SKU2]
    .groupby("week")["Quantity"]
    .sum()
    .reindex(semanas, fill_value=0)
)

serie2.index = pd.DatetimeIndex(semanas)

# Separar entrenamiento y test con los cortes originales
train2 = serie2.loc[semanas_train]
test2 = serie2.loc[semanas_test]

# Descripción del entrenamiento de ambos productos
r3_desc = pd.DataFrame({
    "Producto": ["SKU 1", "SKU 2"],
    "SKU": [SKU, SKU2],
    "Semanas de entrenamiento": [len(train), len(train2)],
    "Semanas con ventas": [(train > 0).sum(), (train2 > 0).sum()],
    "Unidades totales": [train.sum(), train2.sum()],
    "Promedio semanal": [train.mean(), train2.mean()],
    "Mediana semanal": [train.median(), train2.median()],
    "Desviación estándar": [train.std(), train2.std()],
    "Semanas sin ventas": [(train == 0).sum(), (train2 == 0).sum()],
    "Venta máxima semanal": [train.max(), train2.max()]
})

display(r3_desc.round(2))

# Gráfico comparativo: solo entrenamiento
fig = px.line(
    title="Reto 3A: ventas semanales durante el entrenamiento"
)
fig.add_scatter(
    x=train.index, y=train.values,
    mode="lines", name="SKU 1"
)
fig.add_scatter(
    x=train2.index, y=train2.values,
    mode="lines", name=f"SKU 2 ({SKU2})"
)
fig.update_layout(
    xaxis_title="Semana",
    yaxis_title="Unidades vendidas",
    legend_title="Producto"
)
fig.show()

Segundo SKU: 85123A


,StockCode,semanas_con_venta,unidades,primera_venta,ultima_venta,cobertura
1,85123A,94,80903,2009-12-07,2011-10-03,1.0


,Producto,SKU,Semanas de entrenamiento,Semanas con ventas,Unidades totales,Promedio semanal,Mediana semanal,Desviación estándar,Semanas sin ventas,Venta máxima semanal
0,SKU 1,84077,96,93,88273.0,919.51,584.5,1080.36,3,7063.0
1,SKU 2,85123A,96,94,80903.0,842.74,718.5,582.80,2,3329.0


**Interpretación 3A:**

El segundo producto identificado es el SKU **85123A**, con 80,903 unidades vendidas durante las 96 semanas de entrenamiento. Presentó ventas en 94 semanas y un promedio de 842.74 unidades por semana.

Al compararlo con el primer producto, SKU 84077, se observa que este último tuvo un mayor volumen total, con 88,273 unidades, y un promedio semanal superior de 919.51 unidades. Sin embargo, el segundo producto mostró menor variabilidad: su desviación estándar fue de 582.80 unidades, frente a 1,080.36 del primero.

Además, el SKU 85123A tuvo dos semanas sin ventas, mientras que el SKU 84077 tuvo tres. Esto sugiere que el segundo producto presenta una demanda semanal más estable durante el entrenamiento, aunque también registra variaciones y picos de venta.

**Conclusión:** ambos productos tienen niveles y patrones de demanda diferentes. Por ello, no se debe asumir que el modelo seleccionado para el primer producto será el mejor para el segundo. Se debe realizar el backtesting con las mismas reglas y costos para seleccionar el modelo más conveniente para cada SKU.


### Reto 3 · Apartado B — Backtesting y selección con la regla común
Ejecute el mismo backtesting con Naïve, Naïve estacional y Exponential Smoothing (`backtest`), resuma los resultados y seleccione con la regla común. Presente la tabla e indique si la selección se decidió por la tolerancia del 1 %.

In [45]:
# Ejecutar backtesting para el segundo producto
filas2, _ = backtest(train2, MODELOS, "reto3")

# Resumir los resultados y seleccionar con la regla común
tabla2 = resumir(filas2, "modelo", MODELOS)
SELECCIONADO2 = seleccionar(tabla2)

display(tabla2.round(2))

print("Modelo seleccionado para el SKU", SKU2, ":", SELECCIONADO2)

# Comprobar la tolerancia del 1 % respecto al costo mínimo
costo_min2 = tabla2["costo_prom"].min()
limite2 = costo_min2 * 1.01

candidatos2 = tabla2[
    tabla2["costo_prom"] <= limite2 + 1e-9
]

print(f"Costo mínimo: £{costo_min2:.2f}")
print(f"Límite con tolerancia del 1 %: £{limite2:.2f}")
print("Modelos dentro de la tolerancia:", list(candidatos2.index))

,costo_prom,costo_sd,MAE_prom,MAE_sd,RMSE_prom,WAPE_prom,excedentes,faltantes,complejidad
modelo,,,,,,,,,
Naive,4814.82,3483.34,670.62,656.73,867.66,0.97,14694.0,12131.0,0
SeasonalNaive,3306.84,2147.86,459.32,290.28,665.50,0.67,10023.0,8350.0,1
ExpSmoothing,2591.80,1907.11,475.12,266.42,569.80,0.71,14135.0,4870.0,2


Modelo seleccionado para el SKU 85123A : ExpSmoothing
Costo mínimo: £2591.80
Límite con tolerancia del 1 %: £2617.72
Modelos dentro de la tolerancia: ['ExpSmoothing']


**Interpretación 3B:**

El backtesting del SKU 85123A muestra que **Exponential Smoothing fue el modelo seleccionado**, con el menor costo promedio de £2,591.80. Le siguen SeasonalNaive, con £3,306.84, y Naive, con £4,814.82.

El modelo seleccionado también obtuvo el menor RMSE promedio, de 569.80 unidades, y la menor cantidad de faltantes, con 4,870 unidades. Sin embargo, SeasonalNaive presentó un MAE promedio ligeramente menor que Exponential Smoothing: 459.32 frente a 475.12 unidades.

La tolerancia del 1 % establece un costo máximo de £2,617.72. Solo Exponential Smoothing quedó dentro de ese límite, por lo que **la selección se decidió por tener el menor costo, no por un desempate entre modelos dentro de la tolerancia**.

Se selecciona Exponential Smoothing para el SKU 85123A. Esta selección se basa exclusivamente en la validación; el desempeño en test se analizará en el apartado C.


### Reto 3 · Apartado C — Evaluar en test y comparar ambos productos
Evalúe en test el modelo seleccionado y Naïve para el segundo producto. Construya una tabla comparativa de ambos productos (sin colchón) con modelo seleccionado, costo de validación, costo de test del seleccionado y de Naïve, ahorro o sobrecosto, folds en que superó a Naïve y resultado de la evaluación, aplicando la regla de adopción de la sección 13.

In [52]:
# Evaluar el SKU 85123A en test, sin colchón
pronostico_sel2 = pronosticar(SELECCIONADO2, train2, len(test2))
pronostico_naive2 = pronosticar("Naive", train2, len(test2))

# Corregido: eliminar la etiqueta de texto para evitar el conflicto con los parámetros over/short
res_sel2 = evaluar(test2, pronostico_sel2)
res_naive2 = evaluar(test2, pronostico_naive2)

print("Resultados del segundo producto en test:")
display(pd.DataFrame([res_sel2, res_naive2]).round(2))

# Costos de validación del segundo producto
costo_val_sel2 = tabla2.loc[SELECCIONADO2, "costo_prom"]
costo_val_naive2 = tabla2.loc["Naive", "costo_prom"]

# Corregido: convertir la lista filas2 a DataFrame antes de pivotar
df_filas2 = pd.DataFrame(filas2)
costos_fold2 = df_filas2.pivot(index="fold", columns="modelo", values="costo")
folds_ganados2 = int(
    (costos_fold2[SELECCIONADO2] < costos_fold2["Naive"]).sum()
)

# Extraer costos de test del segundo producto
costo_test_sel2 = res_sel2["costo"]
costo_test_naive2 = res_naive2["costo"]
diferencia2 = costo_test_naive2 - costo_test_sel2

print(f"Folds en que {SELECCIONADO2} superó a Naive: {folds_ganados2}/{len(costos_fold2)}")
print(f"Ahorro en test frente a Naive: £{diferencia2:.2f}")

# Consolidar con los resultados del primer producto
# Se reutilizan los resultados calculados en la sección guiada.
costo_val_sel1 = resumen_val.loc[SELECCIONADO, "costo_prom"]
costo_val_naive1 = resumen_val.loc["Naive", "costo_prom"]

costo_test_sel1 = res_test.loc[f"{SELECCIONADO} (sin colchón)", "costo"] if MULT == 0 else res_test.loc[f"Política fijada: {POLITICA}", "costo"]
costo_test_naive1 = res_test.loc["Naive (sin colchón)", "costo"]

costos_fold1 = filas_bt.pivot(index="fold", columns="modelo", values="costo")
folds_ganados1 = int(
    (costos_fold1[SELECCIONADO] < costos_fold1["Naive"]).sum()
)

comparacion = pd.DataFrame([
    {
        "Producto": SKU,
        "Modelo seleccionado": SELECCIONADO,
        "Costo validación seleccionado": costo_val_sel1,
        "Costo test seleccionado": costo_test_sel1,
        "Costo test Naive": costo_test_naive1,
        "Ahorro frente a Naive en test": costo_test_naive1 - costo_test_sel1,
        "Folds ganados frente a Naive": f"{folds_ganados1}/{len(costos_fold1)}"
    },
    {
        "Producto": SKU2,
        "Modelo seleccionado": SELECCIONADO2,
        "Costo validación seleccionado": costo_val_sel2,
        "Costo test seleccionado": costo_test_sel2,
        "Costo test Naive": costo_test_naive2,
        "Ahorro frente a Naive en test": diferencia2,
        "Folds ganados frente a Naive": f"{folds_ganados2}/{len(costos_fold2)}"
    }
])

# Regla de adopción: revisar la sección 13 y usar sus criterios exactos.
display(comparacion.round(2))

Resultados del segundo producto en test:


,MAE,RMSE,WAPE,unidades_excedentes,unidades_faltantes,costo
0,360.25,502.79,0.51,810.0,2072.0,3432.0
1,554.12,717.80,0.78,8.0,4425.0,6640.7


Folds en que ExpSmoothing superó a Naive: 5/5
Ahorro en test frente a Naive: £3208.70


,Producto,Modelo seleccionado,Costo validación seleccionado,Costo test seleccionado,Costo test Naive,Ahorro frente a Naive en test,Folds ganados frente a Naive
0,84077,ExpSmoothing,4324.1,8957.6,7087.9,-1869.7,4/5
1,85123A,ExpSmoothing,2591.8,3432.0,6640.7,3208.7,5/5


### Reto 3 · Apartado D — Recomendación consolidada
Prepare una recomendación para la empresa: qué haría con cada producto (adoptar, mantener en evaluación o no adoptar), qué evidencia la respalda y por qué dos productos no permiten generalizar el método a toda la cartera.


**Interpretación 3C**

En el segundo producto, SKU 85123A, Exponential Smoothing obtuvo un costo de test de £3,432.00, frente a £6,640.70 de Naïve. Esto representa un ahorro de **£3,208.70**, y el modelo seleccionado superó a Naïve en los cinco folds de validación. En test, Exponential Smoothing también registró menor MAE (360.25 frente a 554.12) y menor RMSE (502.79 frente a 717.80).

En cambio, para el primer producto, SKU 84077, Exponential Smoothing tuvo un sobrecosto de £1,869.70 frente a Naïve en test, aunque lo superó en cuatro de los cinco folds de validación.

Los resultados muestran que el mismo modelo no funciona igual para todos los productos. Para el SKU 85123A, la evidencia favorece a Exponential Smoothing tanto en validación como en test. Para el SKU 84077, Naïve fue más económico en test, por lo que no conviene reemplazarlo por Exponential Smoothing bajo los costos definidos. La decisión de adopción debe basarse en la regla establecida en la sección 13, utilizando el test como evaluación retrospectiva y no para cambiar la selección.


### Revisión de advertencias y verificaciones automáticas

Se resumen las advertencias de **todos** los ajustes de Exponential Smoothing (backtesting, calibración, test, reentrenamiento final y retos) y se comprueban las condiciones de validez del flujo.

In [53]:
aj = pd.DataFrame(registro_ajustes)
display(aj.groupby("etapa").agg(ajustes=("ajuste", "count"), con_advertencias=("advertencias", lambda s: int((s > 0).sum())),
                                tipos=("tipos", lambda s: "; ".join(sorted({t for x in s for t in x.split("; ") if t}))))
          .sort_values("ajustes", ascending=False))
mensajes = sorted({m for x in aj["mensajes"] for m in x.split(" | ") if m})
print("Mensajes distintos:", mensajes if mensajes else "ninguno")
print("Especificaciones usadas:", aj["especificacion"].value_counts().to_dict())

verificaciones = {
    "Ningún fold invade el test": all(train.index[o + BT_H - 1] < semanas_test[0] for o in origenes),
    "Colchón de cada fold calibrado solo con semanas anteriores a su origen": bool((pol_folds["ultima_semana_calibracion"] < pol_folds["inicio_validacion"]).all()),
    "Colchón del test calculado solo con el entrenamiento": len(train) == len(semanas_train) and train.index[-1] < semanas_test[0],
    "La tabla final usa el multiplicador fijado": COLCHON_FINAL == colchon(MULT, MAE_FINAL),
    "El costo de la política en test se recalcula con su colchón": np.isclose(costo_pol, evaluar(test, q_politica)["costo"]),
}
for k, v in verificaciones.items():
    print(("OK   " if v else "FALLA"), k)

,ajustes,con_advertencias,tipos
etapa,,,
Reto 2,40,0,
reto2,16,0,
calibración,10,0,
,7,0,
reto3,5,0,
backtesting,5,0,
calibración test,2,0,
calibración final,2,0,
final,1,0,


Mensajes distintos: ninguno
Especificaciones usadas: {'nivel + tendencia + estacionalidad aditivas (m = 4)': 89}
OK    Ningún fold invade el test
OK    Colchón de cada fold calibrado solo con semanas anteriores a su origen
OK    Colchón del test calculado solo con el entrenamiento
OK    La tabla final usa el multiplicador fijado
OK    El costo de la política en test se recalcula con su colchón


### Retroalimentación final

**Pregunta 10.** Indique dos razones por las que el modelo podría fallar en producción aun cuando el backtesting haya sido bueno.

**Respuesta:**


1. **Cambios en la demanda:** pueden aparecer promociones, temporadas, tendencias o pedidos extraordinarios que no estuvieron representados en el entrenamiento ni en el backtesting. Esto puede hacer que los pronósticos pierdan precisión.

2. **Cambios en las condiciones operativas:** pueden ocurrir retrasos de proveedores, quiebres de stock o cambios en los plazos de entrega. Aunque el modelo pronostique correctamente las ventas, la empresa podría no tener suficiente inventario disponible para atender la demanda.


# 15. Ticket de salida

**«El modelo seleccionado antes del test fue __________, con la política __________ (costo promedio de validación £______). En el test costó £______ frente a £______ de Naïve. Mi decisión es __________ (adoptar / mantener en evaluación / no adoptar) porque ____________________. La proyección académica de ______ unidades no es una orden de compra porque ____________________.»**


**Respuesta:**

El modelo seleccionado antes del test fue **Exponential Smoothing**, con la política **sin colchón**, (costo promedio de validación £4,324.10). En el test costó **£8,957.60** frente a **£7,087.90** de Naïve. Mi decisión es **no adoptar Exponential Smoothing para el primer producto y mantener Naïve**, porque el modelo seleccionado tuvo un sobrecosto de £1,869.70 en test, aunque ganó en 4 de los 5 folds de validación. La proyección académica de **9,291 unidades** no es una orden de compra porque se debe considerar el inventario utilizable, los pedidos que llegarán a tiempo y las necesidades pendientes de semanas anteriores.


# 16. Conclusiones

Registre tres conclusiones de carácter técnico y de negocio.


1. **Selección técnica de modelos:** El backtesting permitió seleccionar Exponential Smoothing para ambos productos por su costo promedio de validación. Sin embargo, los resultados del test demostraron que el mejor modelo puede variar según el producto: Naïve tuvo menor costo para el SKU 84077, mientras que Exponential Smoothing funcionó mejor para el SKU 85123A.

2. **Control de costos e inventario:** Añadir un colchón redujo los faltantes, pero aumentó los excedentes y el costo promedio. Con los costos establecidos, la política sin colchón fue la más económica en validación, aunque los faltantes observados en test muestran que se debe seguir evaluando el riesgo de desabastecimiento.

3. **Aplicación al negocio:** Los pronósticos deben utilizarse como apoyo para planificar, no como órdenes de compra automáticas. Antes de comprar, la empresa debe considerar el inventario disponible, los pedidos que llegarán a tiempo y las necesidades acumuladas de semanas anteriores. Además, debe monitorear los resultados y volver a evaluar los modelos cuando cambie la demanda.


# 17. Evidencia de entrega

- [X] Fuente, hojas, dimensiones, rango de fechas y versiones visibles.
- [X] Limpieza documentada (incluidos `StockCode` nulos).
- [X] Semanas de lunes a domingo; semanas incompletas excluidas.
- [X] Test de 8 semanas reservado antes de seleccionar el SKU, el modelo y la política.
- [X] SKU seleccionado con criterios previos al test y límites del criterio de disponibilidad explicados.
- [X] Exploración solo con entrenamiento.
- [X] Backtesting de 8 semanas con fechas de cada fold y comprobación de que no invade el test.
- [X] Regla única de selección aplicada a modelo, política y retos.
- [X] Colchón calibrado solo con errores previos a cada origen.
- [X] Política fijada antes del test y evaluada una vez.
- [X] Decisión de adopción separada de la proyección académica.
- [X] Tres retos individuales con sus apartados A, B, C y D resueltos e interpretados; advertencias revisadas.
- [X] Notebook en GitHub (`semana10/`) y en su propio Colab; ambos URLs presentados en la actividad de la semana.
- [X] Uso de IA declarado.

# Referencias
- Chen, D. (2012). *Online Retail II* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5CG6D
- Hyndman, R. J., & Athanasopoulos, G. (2021). *Forecasting: Principles and Practice* (3rd ed.). OTexts.
- Sharda, R., Delen, D., & Turban, E. *Business Intelligence, Analytics, Data Science and AI*. Pearson.